# 🇸🇦 Arabic Text-to-SQL — Inference-only Ablation Suite (seeded, resumable)

All configurations are evaluated at inference from the same fine-tuned QLoRA adapter (loaded from Google Drive). No fine-tuning cells.

| Configuration | Description |
|---|---|
| `full_system` | Complete pipeline |
| `greedy_control` | Voting OFF, repair OFF (single greedy decode) |
| `abl_no_voting` | − Self-consistency voting |
| `abl_no_repair` | − Repair stage |
| `abl_no_column_linking` | − Column-linking hints |
| `abl_no_value_injection` | − Value injection |
| `abl_no_sample_rows` | − Sample rows |
| `grp_no_schema_grounding` (G1) | − All schema-grounding components |
| `grp_no_value_grounding` (G2) | − All value-grounding components |
| `grp_no_translation` (G3) | − External translation |
| `baseline_minimal` | Raw DDL + question, greedy, no repair |

**How to run.**
1. Run every cell from the top down to *Combined Pipeline*. Set the adapter path in *Load Adapter from Google Drive* if it differs.
2. In *Experiment Suite — Configuration* set `EXP_ROOT`. Smoke-test with `SMOKE_TEST_N = 5`, then set it back to `0`.
3. Run *Experiment Suite — Run selected configurations*. Results are written locally and mirrored to Drive; re-run after a disconnect to resume.
4. *Experiment Suite — Summary & master JSON* builds `ablation_suite_ar.json`.


## 📦 Install Dependencies

In [ ]:
#@title 📦 Install Dependencies { display-mode: "form" }
!pip install -q transformers accelerate peft bitsandbytes datasets
!pip install -q sentencepiece sqlparse nltk sentence-transformers
!pip install -q gdown

"""## 📥 Download Ar-Spider Dataset"""

#@title 📥 Download Ar-Spider Dataset { display-mode: "form" }

# Download and unzip
!gdown --id 1ShZPbM2FvKQy5bh-IpLrtbFNe8LpX-ig
!unzip -qo t2s_datasets.zip

# Rename files to standard names
!mv /content/arspider/Ar_dev_spider.json    /content/arspider/dev.json   2>/dev/null || true
!mv /content/arspider/Ar_train_spider.json  /content/arspider/train.json 2>/dev/null || true

import os,  nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ All dependencies installed")

## 📥 Download Ar-Spider Dataset

In [ ]:
# Download and unzip
!gdown --id 1ShZPbM2FvKQy5bh-IpLrtbFNe8LpX-ig
!unzip -qo t2s_datasets.zip

# Rename files to standard names
!mv /content/arspider/Ar_dev_spider.json    /content/arspider/dev.json   2>/dev/null || true
!mv /content/arspider/Ar_train_spider.json  /content/arspider/train.json 2>/dev/null || true

import os,  nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ All dependencies installed")

AR_SPIDER_DIR = "/content/arspider"

for f in ["train.json", "dev.json", "tables.json"]:
    path = os.path.join(AR_SPIDER_DIR, f)
    assert os.path.exists(path), f"❌ Missing: {path}"
    print(f"   ✅ {f}")

print(f"✅ Ar-Spider dataset ready at {AR_SPIDER_DIR}")

## 📦 Core Imports & Utilities

In [ ]:
import json, re, os, math, gc, sqlite3, time
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import Counter, defaultdict
import sqlparse
from tqdm.auto import tqdm

DATA_DIR = Path("./data"); DATA_DIR.mkdir(exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 🗄️ Schema Loading & SQL Execution

In [ ]:
import glob

def spider_tables_json_to_ddl(tables_json_data: List[Dict]) -> Dict[str, str]:
    """Convert Spider-format tables.json into {db_id: CREATE TABLE DDL}."""
    db_schemas = {}
    for db in tables_json_data:
        db_id = db["db_id"]
        table_names = db["table_names_original"]
        col_names = db["column_names_original"]
        col_types = db.get("column_types", [])
        pks = set(db.get("primary_keys", []))
        fks = db.get("foreign_keys", [])

        table_cols = defaultdict(list)
        for col_idx, (tbl_idx, col_name) in enumerate(col_names):
            if tbl_idx == -1:
                continue
            ctype = col_types[col_idx] if col_idx < len(col_types) else "TEXT"
            sql_type = {
                "text": "TEXT", "number": "REAL", "time": "TEXT",
                "boolean": "INTEGER", "others": "TEXT",
            }.get(ctype.lower(), "TEXT")
            is_pk = col_idx in pks
            table_cols[tbl_idx].append((col_name, sql_type, is_pk))

        fk_map = {}
        for fk_col, ref_col in fks:
            if ref_col < len(col_names):
                ref_tbl_idx, ref_col_name = col_names[ref_col]
                if ref_tbl_idx >= 0:
                    fk_map[fk_col] = (table_names[ref_tbl_idx], ref_col_name)

        ddl_parts = []
        for tbl_idx, tbl_name in enumerate(table_names):
            cols_sql = []
            for col_name, sql_type, is_pk in table_cols.get(tbl_idx, []):
                line = f"    {col_name} {sql_type}"
                if is_pk:
                    line += " PRIMARY KEY"
                cols_sql.append(line)
            fk_lines = []
            for col_idx, (tbl_i, cn) in enumerate(col_names):
                if tbl_i == tbl_idx and col_idx in fk_map:
                    ref_tbl, ref_col = fk_map[col_idx]
                    fk_lines.append(
                        f"    FOREIGN KEY ({cn}) REFERENCES {ref_tbl}({ref_col})"
                    )
            all_lines = cols_sql + fk_lines
            if not all_lines:
                all_lines = ["    id INTEGER PRIMARY KEY"]
            ddl = f"CREATE TABLE {tbl_name} (\n" + ",\n".join(all_lines) + "\n);"
            ddl_parts.append(ddl)

        db_schemas[db_id] = "\n\n".join(ddl_parts)
    return db_schemas


def execute_sql_on_db(db_path: str, sql: str, timeout=30) -> Tuple[bool, Any]:
    """Execute against a real SQLite database file (read-only)."""
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=timeout)
        conn.execute("PRAGMA busy_timeout = 5000")
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description] if cur.description else []
        conn.close()
        return True, {"columns": cols, "rows": rows}
    except Exception as e:
        return False, str(e)


def compare_results(r1, r2) -> bool:
    """Compare execution results as unordered sets."""
    if r1 is None or r2 is None:
        return False
    try:
        rows1 = r1["rows"] if isinstance(r1, dict) else r1
        rows2 = r2["rows"] if isinstance(r2, dict) else r2
        set1 = set(tuple(sorted(str(v) for v in r)) for r in rows1)
        set2 = set(tuple(sorted(str(v) for v in r)) for r in rows2)
        if set1 == set2:
            return True
        if len(rows1) == len(rows2):
            norm1 = sorted(tuple(sorted(str(v) for v in r)) for r in rows1)
            norm2 = sorted(tuple(sorted(str(v) for v in r)) for r in rows2)
            if norm1 == norm2:
                return True
            def _nv(v):
                s = str(v).strip()
                try:
                    f = float(s)
                    return str(int(f)) if f == int(f) else f"{f:.6f}"
                except (ValueError, OverflowError):
                    return s.lower()
            n1 = sorted(tuple(sorted(_nv(v) for v in r)) for r in rows1)
            n2 = sorted(tuple(sorted(_nv(v) for v in r)) for r in rows2)
            return n1 == n2
        return False
    except Exception:
        return str(r1) == str(r2)


# Load schemas
with open(os.path.join(AR_SPIDER_DIR, "tables.json"), "r", encoding="utf-8") as f:
    _tables_json = json.load(f)
ar_spider_schemas = spider_tables_json_to_ddl(_tables_json)
print(f"✅ Loaded schemas: {len(ar_spider_schemas)} databases")

# Find database files
db_paths = {}
db_base = ""
for candidate in ["databases", "database"]:
    p = os.path.join(AR_SPIDER_DIR, candidate)
    if os.path.isdir(p):
        db_base = p
        break
if db_base:
    for db_name in os.listdir(db_base):
        db_dir = os.path.join(db_base, db_name)
        if os.path.isdir(db_dir):
            for ext in ["*.sqlite", "*.db", "*.sqlite3"]:
                found = glob.glob(os.path.join(db_dir, ext))
                if found:
                    db_paths[db_name] = found[0]
                    break
print(f"✅ Found {len(db_paths)} SQLite database files")

## 📏 Evaluation Metrics (BLEU, SQAM, TSED, EM, EX)

In [ ]:
def tokenize_sql(sql: str) -> List[str]:
    s = sql.strip().rstrip(";").lower()
    s = re.sub(r'([(),=<>!+\-*/])', r' \1 ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s.split()

def compute_bleu(pred_sql: str, gold_sql: str, max_n: int = 4) -> float:
    pred_tokens = tokenize_sql(pred_sql)
    gold_tokens = tokenize_sql(gold_sql)
    if not pred_tokens or not gold_tokens:
        return 0.0
    bp = min(1.0, math.exp(1 - len(gold_tokens) / max(len(pred_tokens), 1)))
    log_avg = 0.0
    for n in range(1, max_n + 1):
        pred_ng = Counter(tuple(pred_tokens[i:i+n]) for i in range(len(pred_tokens) - n + 1))
        gold_ng = Counter(tuple(gold_tokens[i:i+n]) for i in range(len(gold_tokens) - n + 1))
        clipped = sum(min(pred_ng[ng], gold_ng[ng]) for ng in pred_ng)
        total = max(sum(pred_ng.values()), 1)
        if clipped == 0 and n > 1:
            clipped = 1
        precision = clipped / total
        log_avg += (1.0 / max_n) * math.log(max(precision, 1e-10))
    return bp * math.exp(log_avg)

SQAM_WEIGHTS = {
    "select": 0.30, "from": 0.20, "where": 0.25,
    "group_by": 0.10, "having": 0.05, "order_by": 0.05, "limit": 0.05,
}

def _extract_clause(sql, clause, next_clauses):
    pattern = rf'\b{clause}\b\s+(.*?)(?={"|".join(rf"(?:\b{nc}\b)" for nc in next_clauses)}|;|$)'
    m = re.search(pattern, sql, re.I | re.DOTALL)
    return m.group(1).strip() if m else ""

def _parse_sql_clauses(sql):
    s = re.sub(r'\s+', ' ', sql.strip().rstrip(";").strip())
    clauses = {}
    clause_order = ["select", "from", "where", "group by", "having", "order by", "limit"]
    next_map = {
        "select": ["from","where","group by","having","order by","limit"],
        "from": ["where","group by","having","order by","limit"],
        "where": ["group by","having","order by","limit"],
        "group by": ["having","order by","limit"],
        "having": ["order by","limit"],
        "order by": ["limit"],
        "limit": [],
    }
    for cl in clause_order:
        nxt = next_map.get(cl, [])
        content = _extract_clause(s, cl.replace(" ",r"\s+"), [nc.replace(" ",r"\s+") for nc in nxt])
        clauses[cl.replace(" ","_")] = content
    return clauses

def _split_items(clause_content):
    if not clause_content: return set()
    items, depth, cur = [], 0, ""
    for ch in clause_content:
        if ch == '(': depth += 1
        elif ch == ')': depth -= 1
        if ch == ',' and depth == 0:
            items.append(cur.strip().lower()); cur = ""
        else: cur += ch
    if cur.strip(): items.append(cur.strip().lower())
    expanded = []
    for item in items:
        parts = re.split(r'\b(?:and|or)\b', item, flags=re.I)
        expanded.extend(p.strip() for p in parts if p.strip())
    return set(expanded)

def compute_sqam(pred_sql, gold_sql):
    pred_c = _parse_sql_clauses(pred_sql)
    gold_c = _parse_sql_clauses(gold_sql)
    tw, ws = 0.0, 0.0
    for cn, w in SQAM_WEIGHTS.items():
        pi = _split_items(pred_c.get(cn,""))
        gi = _split_items(gold_c.get(cn,""))
        if not gi and not pi: continue
        tw += w
        if not gi or not pi: continue
        inter = pi & gi
        p = len(inter)/len(pi) if pi else 0
        r = len(inter)/len(gi) if gi else 0
        f1 = (2*p*r)/(p+r) if (p+r) > 0 else 0
        ws += w * f1
    return ws / tw if tw > 0 else 0.0

class SimpleNode:
    def __init__(self, label, children=None):
        self.label = label; self.children = children or []

def _sqlparse_to_node(token):
    if hasattr(token, 'tokens'):
        label = str(token.ttype or type(token).__name__).split('.')[-1]
        return SimpleNode(label, [_sqlparse_to_node(t) for t in token.tokens if str(t).strip()])
    return SimpleNode(str(token).strip().lower() or "EMPTY")

def _count_nodes(n): return 1 + sum(_count_nodes(c) for c in n.children)

def _tree_sim(t1, t2):
    score = 1.0 if t1.label == t2.label else 0.0
    if not t1.children and not t2.children: return score
    matched, used = 0, set()
    for c1 in t1.children:
        best_s, best_j = 0, -1
        for j, c2 in enumerate(t2.children):
            if j in used: continue
            s = _tree_sim(c1, c2)
            if s > best_s: best_s, best_j = s, j
        if best_j >= 0: matched += best_s; used.add(best_j)
    total = max(len(t1.children), len(t2.children))
    return 0.4 * score + 0.6 * (matched / total if total else 1.0)

def compute_tsed(pred_sql, gold_sql):
    try:
        pp = sqlparse.parse(pred_sql.strip())
        gp = sqlparse.parse(gold_sql.strip())
        if not pp or not gp: return 0.0
        return _tree_sim(_sqlparse_to_node(pp[0]), _sqlparse_to_node(gp[0]))
    except:
        pt = set(tokenize_sql(pred_sql)); gt = set(tokenize_sql(gold_sql))
        return len(pt & gt) / max(len(pt | gt), 1) if gt else 0.0

def _normalize_sql_for_em(sql):
    s = sql.strip().rstrip(";").strip().lower()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[`"\[\]]', '', s)
    s = re.sub(r'\(\s+', '(', s)
    s = re.sub(r'\s+\)', ')', s)
    s = re.sub(r'\bas\s+(?!integer|real|text|numeric|blob|int|varchar|char|float|double|boolean|date|time|datetime)\w+', '', s)
    return s.strip()

def compute_exact_match(pred_sql, gold_sql):
    return _normalize_sql_for_em(pred_sql) == _normalize_sql_for_em(gold_sql)

print("✅ Evaluation metrics loaded (EX, EM, BLEU, SQAM, TSED)")

## ⚙️ Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#@title ⚙️ Configuration { display-mode: "form" }

FINETUNE_7B_BASE  = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LENGTH_7B = 4096      #@param {type:"integer"}   # prompt length cap at inference
USE_ENHANCED_PROMPT = True     #@param {type:"boolean"}


# ── Prompt template (SQL-only, no CoT) ────────────────────────
SQL_ONLY_PROMPT = '''Database Engine: SQLite

Database Schema:
{db_details}

Question:
{question}

Generate the SQL query that answers this question. Output ONLY the SQL inside a code block:
```sql
-- Your SQL query
```'''

SQL_ONLY_SYSTEM = (
    "You are an expert Arabic-to-SQL translator. Given a database schema "
    "and a question (in Arabic or English), carefully map Arabic terms to "
    "the correct English column and table names in the schema, then output "
    "ONLY the correct SQL query inside a code block. No explanations."
)

# ── Enhanced prompt template (with table summary + JOIN guidance) ─
SQL_ENHANCED_PROMPT = '''Database Engine: SQLite

Database Schema:
{db_details}

Table → Column Reference (use ONLY columns from the table that has them):
{table_summary}

IMPORTANT: Use the MINIMUM number of tables needed.
If ALL required columns exist in ONE table, do NOT use JOIN.

Question:
{question}

Generate the SQL query that answers this question. Output ONLY the SQL inside a code block:
```sql
-- Your SQL query
```'''


def build_table_column_summary(schema_ddl: str) -> str:
    """Generate a compact table→columns mapping for the enhanced prompt."""
    tables = {}
    cur_table = None
    for line in schema_ddl.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I,
        )
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+\w+', line)
            if cm and not any(
                line.upper().startswith(kw) for kw in
                ('PRIMARY', 'FOREIGN', 'UNIQUE', 'CHECK', 'CONSTRAINT', '--')
            ):
                tables[cur_table].append(cm.group(1))
    if not tables:
        return ""
    lines = []
    for table, cols in tables.items():
        lines.append(f"  • {table}: {', '.join(cols)}")
    return "\n".join(lines)


def format_prompt(schema: str, question: str) -> str:
    """Format prompt using the configured template (base or enhanced)."""
    if USE_ENHANCED_PROMPT:
        table_summary = build_table_column_summary(schema)
        return SQL_ENHANCED_PROMPT.format(
            db_details=schema[:4500],
            table_summary=table_summary,
            question=question,
        )
    else:
        return SQL_ONLY_PROMPT.format(
            db_details=schema[:5000],
            question=question,
        )

print(f"✅ Config ready")
print(f"   Base: {FINETUNE_7B_BASE}")
print(f"   Prompt: {'🆕 Enhanced (table summary + JOIN guidance)' if USE_ENHANCED_PROMPT else '📋 Base (schema only)'}")

---
## 🚀 Section A: Tier 1 Feature Configuration (Inference)

Technique toggles used by the inference prompt builder and the ablation suite.

In [ ]:
#@title 🚀 Tier 1 Feature Configuration (Inference) { display-mode: "form" }

# ── Technique Toggles (inference prompt builder) ────────────────
ENABLE_MSCHEMA            = True   #@param {type:"boolean"}
ENABLE_VALUE_HINTS        = True   #@param {type:"boolean"}
VALUE_HINT_MAX_PER_COL    = 4      #@param {type:"integer"}

ENABLE_COLUMN_LINKING     = True   #@param {type:"boolean"}
COL_LINK_TOP_K            = 6      #@param {type:"integer"}

ENABLE_AR_COL_DESC        = True   #@param {type:"boolean"}
AR_COL_DESC_PATH          = "/content/drive/MyDrive/ar_column_descriptions.json"  #@param {type:"string"}

ENABLE_ENGLISH_HINT       = True   #@param {type:"boolean"}

ENABLE_BILINGUAL_SCHEMA   = True   #@param {type:"boolean"}
AR_TABLE_GLOSS_PATH       = "/content/drive/MyDrive/ar_table_glosses.json"  #@param {type:"string"}

ENABLE_VALUE_INJECTION    = True   #@param {type:"boolean"}
VALUE_INJECTION_TOP_K     = 5      #@param {type:"integer"}
VALUE_INJECTION_THRESHOLD = 0.40   #@param {type:"number"}

ENABLE_BILINGUAL_EMBEDDINGS = True   #@param {type:"boolean"}

ENABLE_SAMPLE_ROWS        = True   #@param {type:"boolean"}
SAMPLE_ROWS_LIMIT         = 3      #@param {type:"integer"}
SAMPLE_ROWS_MAX_TABLES    = 3      #@param {type:"integer"}

# ── Inference-only settings (not used during training) ──────────
ENABLE_SELF_CONSISTENCY   = True   #@param {type:"boolean"}
SC_NUM_CANDIDATES         = 8      #@param {type:"integer"}
SC_TEMPERATURE            = 0.7    #@param {type:"number"}
SC_TOP_P                  = 0.95   #@param {type:"number"}
ENABLE_MULTI_TEMP         = True   #@param {type:"boolean"}
SC_TEMP_TIERS = [
    (0.3, 2),   # 2 conservative candidates
    (0.7, 3),   # 3 medium candidates
    (1.1, 2),   # 2 creative candidates
]


# ── Reproducibility additions (behavior-neutral) ─────────────────
SC_DETERMINISTIC_SEEDS    = True   #@param {type:"boolean"}
SC_BASE_SEED              = 1234   #@param {type:"integer"}
LOG_CANDIDATES            = True   #@param {type:"boolean"}

ENABLE_SELF_CORRECTION    = True   #@param {type:"boolean"}
SC_MAX_RETRIES            = 2      #@param {type:"integer"}
ENABLE_ID_FIXING          = True   #@param {type:"boolean"}
ENABLE_FILTER_REMOVAL     = True   #@param {type:"boolean"}
ENABLE_SQL_VALUE_GROUNDING = True   #@param {type:"boolean"}
SQL_VALUE_SIM_THRESHOLD    = 0.70   #@param {type:"number"}
ENABLE_LOW_CONF_FALLBACK  = True   #@param {type:"boolean"}
LOW_CONF_VOTE_THRESHOLD   = 3      #@param {type:"integer"}
LOW_CONF_EXTRA_CANDIDATES = 8      #@param {type:"integer"}
SC_BATCH_SIZE             = 1024   #@param {type:"integer"}
RESUME_FROM               = 0      #@param {type:"integer"}
REGENERATE_DESCRIPTIONS   = False  #@param {type:"boolean"}
REGENERATE_TABLE_GLOSSES  = False  #@param {type:"boolean"}

print("🚀 Tier 1 Feature Configuration:")
print(f"   M-Schema                      : {'✅ ON' if ENABLE_MSCHEMA else '❌ OFF'}")
print(f"   Value Hints                   : {'✅ ON' if ENABLE_VALUE_HINTS else '❌ OFF'}")
print(f"   Column Linking (embedding)    : {'✅ ON' if ENABLE_COLUMN_LINKING else '❌ OFF'}")
print(f"   Arabic Column Descriptions    : {'✅ ON' if ENABLE_AR_COL_DESC else '❌ OFF'}")
print(f"   English Translation Hint      : {'✅ ON' if ENABLE_ENGLISH_HINT else '❌ OFF'}")
print(f"   Bilingual Schema (AR gloss)   : {'✅ ON' if ENABLE_BILINGUAL_SCHEMA else '❌ OFF'}")
print(f"   Value Injection (emb match)   : {'✅ ON' if ENABLE_VALUE_INJECTION else '❌ OFF'}")
print(f"   Bilingual Embeddings          : {'✅ ON' if ENABLE_BILINGUAL_EMBEDDINGS else '❌ OFF'}")
print(f"   Sample Rows in Prompt         : {'✅ ON' if ENABLE_SAMPLE_ROWS else '❌ OFF'}")

In [ ]:
#@title ✅ Documented Full Pipeline Sanity Check { display-mode: "form" }
# Asserts ALL components are ON (this is the headline configuration).
_off = [f for f in ["ENABLE_SELF_CORRECTION","ENABLE_ID_FIXING","ENABLE_FILTER_REMOVAL",
                    "ENABLE_SQL_VALUE_GROUNDING","ENABLE_LOW_CONF_FALLBACK","ENABLE_MULTI_TEMP"]
        if not globals().get(f, False)]
if _off:
    raise RuntimeError("DOCUMENTED-FULL MODE VIOLATION — these must be True: " + ", ".join(_off))
print("✅ Documented full pipeline verified — all components ACTIVE:")
print("   5 prompt components + multi-temp voting + repair stage + low-conf fallback")
print(f"   Deterministic seeds: {'ON (base='+str(SC_BASE_SEED)+')' if SC_DETERMINISTIC_SEEDS else 'OFF'}")
print(f"   Per-candidate logging: {'ON' if LOG_CANDIDATES else 'OFF'}")


---
## 🧰 Section B: Shared Function Definitions (Train + Inference)

Functions used by the inference prompt builder: schema parsing, M-Schema, column linking, value hints, value injection, sample rows, prompt templates.

In [ ]:
#@title 🧰 Shared: Schema Parsing, M-Schema, Lookups { display-mode: "form" }

# Paste as a NEW CELL after Section A.
# These functions are used by BOTH training prompt construction
# and inference. They are identical to the originals in your
# notebook — just defined earlier so training can use them.
# ═══════════════════════════════════════════════════════════════════



#@title 🧰 Shared Functions (Train + Inference) { display-mode: "form" }

import numpy as np
from collections import defaultdict
from dataclasses import dataclass, field

# ── EvalSample dataclass (reused for both train and eval) ───────
@dataclass
class EvalSample:
    id: str
    db_id: str
    arabic_question: str
    gold_sql: str
    schema_ddl: str
    db_path: str = ""
    english_question: str = ""


# ── Schema parsing ──────────────────────────────────────────────

def extract_tables_columns(schema_sql: str) -> Dict[str, List[str]]:
    """Parse CREATE TABLE DDL into {table_name: [column_names]}."""
    tables = {}
    cur_table = None
    for line in schema_sql.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I,
        )
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+\w+', line)
            if cm and not any(
                line.upper().startswith(kw) for kw in
                ('PRIMARY', 'FOREIGN', 'UNIQUE', 'CHECK', 'CONSTRAINT', '--')
            ):
                tables[cur_table].append(cm.group(1))
    return tables


def build_table_column_summary(schema_ddl: str) -> str:
    """Compact table→columns mapping for prompt."""
    tables = extract_tables_columns(schema_ddl)
    if not tables:
        return ""
    lines = []
    for table, cols in tables.items():
        lines.append(f"  • {table}: {', '.join(cols)}")
    return "\n".join(lines)


# ── Index tables.json by db_id ──────────────────────────────────
_tables_json_by_id = {}
try:
    for _entry in _tables_json:
        _tables_json_by_id[_entry["db_id"]] = _entry
    print(f"✅ Indexed {len(_tables_json_by_id)} database schemas for M-Schema")
except NameError:
    print("⚠️ _tables_json not found — will fall back to DDL-based M-Schema")


# ── Arabic description + gloss lookups (populated in Section D) ─
_ar_col_descriptions = {}  # {db_id: {"table.column": "وصف عربي", ...}}
_ar_table_glosses    = {}  # {db_id: {"table_name": "وصف عربي", ...}}


def get_column_description(db_id: str, table: str, column: str) -> str:
    """Look up Arabic description for a column. Returns '' if not found."""
    return _ar_col_descriptions.get(db_id, {}).get(f"{table}.{column}", "")


def get_table_gloss(db_id: str, table_name: str) -> str:
    """Look up Arabic gloss for a table. Returns '' if not found."""
    return _ar_table_glosses.get(db_id, {}).get(table_name, "")


# ── M-Schema builder ───────────────────────────────────────────

def build_mschema(tables_json_entry: dict, db_id: str = "") -> str:
    """Build M-Schema from a single tables.json database entry.
    Includes Arabic table glosses and column descriptions inline."""
    table_names = tables_json_entry["table_names_original"]
    col_names   = tables_json_entry["column_names_original"]
    col_types   = tables_json_entry.get("column_types", [])
    pks         = set(tables_json_entry.get("primary_keys", []))
    fks         = tables_json_entry.get("foreign_keys", [])

    table_cols = defaultdict(list)
    for col_idx, (tbl_idx, col_name) in enumerate(col_names):
        if tbl_idx == -1:
            continue
        ctype = col_types[col_idx] if col_idx < len(col_types) else "text"
        sql_type = {"text": "TEXT", "number": "REAL", "time": "TEXT",
                    "boolean": "INT", "others": "TEXT"}.get(ctype.lower(), "TEXT")
        is_pk = col_idx in pks
        table_cols[tbl_idx].append((col_name, sql_type, is_pk, col_idx))

    fk_map = {}
    for fk_col, ref_col in fks:
        if ref_col < len(col_names):
            ref_tbl_idx, ref_col_name = col_names[ref_col]
            if ref_tbl_idx >= 0:
                fk_map[fk_col] = (table_names[ref_tbl_idx], ref_col_name)

    lines = []
    for tbl_idx, tbl_name in enumerate(table_names):
        cols = table_cols.get(tbl_idx, [])
        if not cols:
            continue
        col_parts = []
        for col_name, sql_type, is_pk, col_idx in cols:
            pk_mark = "*" if is_pk else ""
            ar_desc = ""
            if ENABLE_BILINGUAL_SCHEMA and db_id:
                ar_desc = get_column_description(db_id, tbl_name, col_name)
            if ar_desc:
                col_parts.append(f"({col_name}{pk_mark}, {sql_type}, {ar_desc})")
            else:
                col_parts.append(f"({col_name}{pk_mark}, {sql_type})")
        tbl_gloss = ""
        if ENABLE_BILINGUAL_SCHEMA and db_id:
            tbl_gloss = get_table_gloss(db_id, tbl_name)
        if tbl_gloss:
            lines.append(f"【{tbl_name}】 ({tbl_gloss})")
        else:
            lines.append(f"【{tbl_name}】")
        lines.append("  " + "  ".join(col_parts))
        for col_name, sql_type, is_pk, col_idx in cols:
            if col_idx in fk_map:
                ref_tbl, ref_col = fk_map[col_idx]
                lines.append(f"  -> FK: {col_name} -> {ref_tbl}.{ref_col}")
    return "\n".join(lines)


def build_mschema_from_ddl(schema_ddl: str) -> str:
    """Fallback: build M-Schema from DDL string (no bilingual annotations)."""
    tables = {}
    cur_table = None
    fks_list = []
    for line in schema_ddl.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I)
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            fk_m = re.match(
                r'FOREIGN\s+KEY\s*\((\w+)\)\s*REFERENCES\s+(\w+)\s*\((\w+)\)',
                line, re.I)
            if fk_m:
                fks_list.append((cur_table, fk_m.group(1), fk_m.group(2), fk_m.group(3)))
                continue
            if any(line.upper().startswith(kw) for kw in
                   ('PRIMARY','FOREIGN','UNIQUE','CHECK','CONSTRAINT','--')):
                continue
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+(\w+)', line)
            if cm:
                col_name = cm.group(1)
                col_type = cm.group(2).upper()
                is_pk = 'PRIMARY KEY' in line.upper()
                tables[cur_table].append((col_name, col_type, is_pk))

    lines = []
    for tbl_name, cols in tables.items():
        col_parts = []
        for col_name, col_type, is_pk in cols:
            pk_mark = "*" if is_pk else ""
            col_parts.append(f"({col_name}{pk_mark}, {col_type})")
        lines.append(f"【{tbl_name}】")
        lines.append("  " + "  ".join(col_parts))
        for fk_tbl, fk_col, ref_tbl, ref_col in fks_list:
            if fk_tbl == tbl_name:
                lines.append(f"  -> FK: {fk_col} -> {ref_tbl}.{ref_col}")
    return "\n".join(lines)


def get_mschema(db_id: str, schema_ddl: str) -> str:
    """Get M-Schema for a database (from tables.json if available, else DDL)."""
    if db_id in _tables_json_by_id:
        return build_mschema(_tables_json_by_id[db_id], db_id=db_id)
    return build_mschema_from_ddl(schema_ddl)

In [ ]:
#@title 🧰 Shared: Value Hints, Column Linking, Value Injection, Sample Rows { display-mode: "form" }

# ── Database value hints (question-aware) ───────────────────────

def sample_db_values_question_aware(db_path: str, schema_ddl: str,
                                     question: str,
                                     max_per_col: int = 4) -> str:
    """Question-aware value sampling from real DB. Returns compact hint string."""
    if not db_path or not os.path.exists(db_path):
        return ""
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return ""

    all_hints = []
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl_name, columns in schema_tables.items():
            for col_name in columns:
                try:
                    cursor.execute(f'PRAGMA table_info("{tbl_name}")')
                    col_info = {row[1].lower(): row[2] for row in cursor.fetchall()}
                    col_type = col_info.get(col_name.lower(), "TEXT").upper()
                    is_text = any(t in col_type for t in
                                  ["TEXT", "VARCHAR", "CHAR", "CLOB"])
                    if not is_text:
                        continue
                    cursor.execute(
                        f'SELECT DISTINCT "{col_name}" FROM "{tbl_name}" '
                        f'WHERE "{col_name}" IS NOT NULL AND TRIM("{col_name}") != "" '
                        f'LIMIT {max_per_col * 2}')
                    values = [str(row[0]) for row in cursor.fetchall()
                              if row[0] is not None and str(row[0]).strip()]
                    if not values:
                        continue
                    col_lower = col_name.lower()
                    relevance = 0
                    high_value = ['name', 'type', 'status', 'country', 'city',
                                  'state', 'region', 'continent', 'language',
                                  'nationality', 'category', 'genre', 'major',
                                  'department', 'sex', 'gender', 'color',
                                  'brand', 'title', 'position', 'head']
                    if any(hv in col_lower for hv in high_value):
                        relevance += 2
                    for v in values:
                        if v.lower() in question.lower():
                            relevance += 5
                    display = [v[:35] for v in values[:max_per_col]]
                    all_hints.append((relevance, f"    {tbl_name}.{col_name}: {display}"))
                except Exception:
                    continue
        conn.close()
    except Exception:
        return ""

    if not all_hints:
        return ""
    all_hints.sort(key=lambda x: -x[0])
    selected = [h[1] for h in all_hints[:min(len(all_hints), 15)]]
    return "Sample column values (use these EXACT values in your SQL):\n" + "\n".join(selected)


# ── Column linking (embedding similarity) ──────────────────────
# Requires _emb_model — loaded in Section D.  Functions degrade
# gracefully to no-ops when _emb_model is None.

_emb_model    = None   # set in Section D
_col_emb_cache = {}    # {db_id: {"candidates": [...], "embeddings": np.ndarray}}


def _build_column_description(table_name: str, col_name: str,
                               db_id: str = "") -> str:
    """Build natural-language description for embedding.
    Includes Arabic description for bilingual matching."""
    readable = re.sub(r'([a-z])([A-Z])', r'\1 \2', col_name)
    readable = readable.replace('_', ' ').lower()
    tbl_readable = re.sub(r'([a-z])([A-Z])', r'\1 \2', table_name)
    tbl_readable = tbl_readable.replace('_', ' ').lower()
    base = f"{readable} in {tbl_readable}"
    if ENABLE_BILINGUAL_EMBEDDINGS and db_id:
        ar_desc = get_column_description(db_id, table_name, col_name)
        if ar_desc:
            return f"{base} — {ar_desc}"
    return base


def _get_column_embeddings(db_id: str, schema_ddl: str) -> dict:
    """Get or compute cached column embeddings for a database."""
    if db_id in _col_emb_cache:
        return _col_emb_cache[db_id]

    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables or _emb_model is None:
        empty = {"candidates": [], "embeddings": np.array([])}
        _col_emb_cache[db_id] = empty
        return empty

    candidates = []
    texts = []
    for tbl, cols in schema_tables.items():
        for col in cols:
            desc = _build_column_description(tbl, col, db_id=db_id)
            candidates.append({
                "ref": f"{tbl}.{col}", "text": desc,
                "tbl": tbl, "col": col,
            })
            texts.append(desc)

    if texts:
        embeddings = _emb_model.encode(
            texts, normalize_embeddings=True,
            show_progress_bar=False, batch_size=64,
        )
    else:
        embeddings = np.array([])

    result = {"candidates": candidates, "embeddings": embeddings}
    _col_emb_cache[db_id] = result
    return result


def link_columns_to_question(db_id: str, schema_ddl: str, question: str,
                              db_path: str = "", top_k: int = 6) -> str:
    """Score every column against the Arabic question using embedding similarity.
    Returns a prompt hint with the top-K most relevant columns."""
    if _emb_model is None:
        return ""

    col_data = _get_column_embeddings(db_id, schema_ddl)
    candidates = col_data["candidates"]
    col_embeddings = col_data["embeddings"]

    if not candidates or len(col_embeddings) == 0:
        return ""

    q_embedding = _emb_model.encode(
        [question], normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    final_scores = col_embeddings @ q_embedding

    ranked_indices = np.argsort(-final_scores)[:top_k]
    lines = []
    for idx in ranked_indices:
        if final_scores[idx] < 0.50:  # raised to 0.50 — only strong matches survive
            continue
        c = candidates[idx]
        ar_desc = ""
        if ENABLE_AR_COL_DESC:
            ar_desc = get_column_description(db_id, c["tbl"], c["col"])
        if ar_desc:
            lines.append(f"  ★ {c['ref']} — {ar_desc}")
        else:
            lines.append(f"  ★ {c['ref']}")

    if not lines:
        return ""
    return "Likely relevant columns (prefer these):\n" + "\n".join(lines)


# ── Question-matched value injection (embedding-based) ─────────

_db_value_emb_cache = {}  # {db_id: {"values": [...], "embeddings": np.ndarray, "col_refs": [...]}}


def _build_value_embeddings(db_id: str, db_path: str, schema_ddl: str) -> dict:
    """Pre-compute embeddings for all text values in a database."""
    if db_id in _db_value_emb_cache:
        return _db_value_emb_cache[db_id]

    empty = {"values": [], "embeddings": np.array([]), "col_refs": []}
    if not db_path or not os.path.exists(db_path) or _emb_model is None:
        _db_value_emb_cache[db_id] = empty
        return empty

    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        _db_value_emb_cache[db_id] = empty
        return empty

    all_values = []
    col_refs = []
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl, cols in schema_tables.items():
            for col in cols:
                try:
                    cursor.execute(f'PRAGMA table_info("{tbl}")')
                    col_info = {row[1].lower(): row[2] for row in cursor.fetchall()}
                    col_type = col_info.get(col.lower(), "TEXT").upper()
                    if not any(t in col_type for t in ["TEXT", "VARCHAR", "CHAR", "CLOB"]):
                        continue
                    cursor.execute(
                        f'SELECT DISTINCT "{col}" FROM "{tbl}" '
                        f'WHERE "{col}" IS NOT NULL AND TRIM("{col}") != "" '
                        f'LIMIT 150')
                    for row in cursor.fetchall():
                        val = str(row[0]).strip()
                        if len(val) >= 2:
                            all_values.append(val)
                            col_refs.append(f"{tbl}.{col}")
                except Exception:
                    continue
        conn.close()
    except Exception:
        _db_value_emb_cache[db_id] = empty
        return empty

    if not all_values:
        _db_value_emb_cache[db_id] = empty
        return empty

    # Deduplicate
    seen = {}
    unique_values = []
    unique_refs = []
    for val, ref in zip(all_values, col_refs):
        key = val.lower()
        if key not in seen:
            seen[key] = len(unique_values)
            unique_values.append(val)
            unique_refs.append(ref)

    embeddings = _emb_model.encode(
        unique_values, normalize_embeddings=True,
        show_progress_bar=False, batch_size=128,
    )

    result = {"values": unique_values, "embeddings": embeddings, "col_refs": unique_refs}
    _db_value_emb_cache[db_id] = result
    return result


def find_question_matched_values(db_id: str, db_path: str, schema_ddl: str,
                                  arabic_question: str, english_question: str = "",
                                  top_k: int = 5, threshold: float = 0.40) -> str:
    """Find DB values semantically closest to the question.
    Uses BOTH Arabic and English queries for dual-language matching."""
    if _emb_model is None:
        return ""

    val_data = _build_value_embeddings(db_id, db_path, schema_ddl)
    values = val_data["values"]
    val_embeddings = val_data["embeddings"]
    col_refs = val_data["col_refs"]

    if not values or len(val_embeddings) == 0:
        return ""

    ar_emb = _emb_model.encode(
        [arabic_question], normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    sims = val_embeddings @ ar_emb

    if english_question:
        en_emb = _emb_model.encode(
            [english_question], normalize_embeddings=True,
            show_progress_bar=False,
        )[0]
        en_sims = val_embeddings @ en_emb
        sims = np.maximum(sims, en_sims)

    top_indices = np.argsort(-sims)[:top_k]
    hints = []
    seen_vals = set()
    for idx in top_indices:
        if sims[idx] < threshold:
            break
        val = values[idx]
        if val.lower() in seen_vals:
            continue
        seen_vals.add(val.lower())
        col_ref = col_refs[idx]
        hints.append(f"  → {col_ref} = '{val}'")

    if not hints:
        return ""
    return "Detected values in question (use EXACT spelling in WHERE):\n" + "\n".join(hints)


# ── Sample rows in prompt ───────────────────────────────────────

def get_sample_rows(db_path: str, schema_ddl: str, column_hints: str = "",
                     limit: int = 3, max_tables: int = 3) -> str:
    """Get sample rows from the most relevant tables."""
    if not db_path or not os.path.exists(db_path):
        return ""
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return ""

    priority_tables = []
    if column_hints:
        for tbl in schema_tables:
            if tbl in column_hints:
                priority_tables.append(tbl)
    for tbl in schema_tables:
        if tbl not in priority_tables:
            priority_tables.append(tbl)
    selected_tables = priority_tables[:max_tables]

    lines = []
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl in selected_tables:
            cols = schema_tables.get(tbl, [])
            if not cols:
                continue
            try:
                display_cols = cols[:6]
                col_str = ', '.join(f'"{c}"' for c in display_cols)
                cursor.execute(f'SELECT {col_str} FROM "{tbl}" LIMIT {limit}')
                rows = cursor.fetchall()
                if not rows:
                    continue
                header = " | ".join(c[:15] for c in display_cols)
                lines.append(f"  {tbl}: {header}")
                for row in rows:
                    row_str = " | ".join(
                        str(v)[:15] if v is not None else "NULL"
                        for v in row
                    )
                    lines.append(f"    {row_str}")
            except Exception:
                continue
        conn.close()
    except Exception:
        return ""

    if not lines:
        return ""
    return "Sample data (actual rows from database):\n" + "\n".join(lines)

In [ ]:
#@title 🧰 Shared: Prompt Templates & Builders { display-mode: "form" }

# ── Prompt templates ───────────────────────────────────────────

TIER1_SYSTEM = SQL_ONLY_SYSTEM  # reuse the proven system prompt

TIER1_PROMPT = '''Database Engine: SQLite

Database Schema:
{schema}

{value_hints}
{matched_values}
{column_hints}
{sample_rows}
Table -> Column Reference (use ONLY columns from the table that has them):
{table_summary}

IMPORTANT: Use the MINIMUM number of tables needed.
If ALL required columns exist in ONE table, do NOT use JOIN.
Only JOIN tables when the question requires data from multiple tables.
Use the exact column values shown above when filtering.

Question:
{question}
{english_hint}
Generate the SQL query. Output ONLY SQL inside a code block:
```sql
```'''

TIER1_RETRY_PROMPT = '''Database Engine: SQLite

Database Schema:
{schema}

{value_hints}
Your previous SQL had an error:
  SQL: {failed_sql}
  Error: {error_msg}

Question:
{question}

Fix the SQL. Output ONLY the corrected SQL inside a code block:
```sql
```'''





# ── Inference prompt builder (unchanged logic) ──────────────────



def _question_has_text_entities(question: str, english_question: str = "") -> bool:
    """Check if the question likely needs text value matching.
    Returns False for pure numeric/count/aggregation queries where
    value injection would just add noise.

    Heuristic: True if the question contains proper nouns, quoted strings,
    or named entities that could match DB text values."""
    combined = question + " " + english_question

    # Check for quoted strings in the question
    if re.search(r'["\'].+?["\'"]', combined):
        return True

    # Check for Latin proper nouns (capitalized words that aren't SQL keywords)
    sql_words = {'select','from','where','join','and','or','not','in','the',
                 'what','which','how','many','show','find','list','give','all',
                 'is','are','was','were','has','have','had','does','did',
                 'who','when','with','for','that','this','than','then',
                 'more','less','each','every','any','some','no','by',
                 'on','at','to','of','a','an','do','be','it','its',
                 'count','average','maximum','minimum','total','number',
                 'most','least','highest','lowest','largest','smallest',
                 'between','above','below','greater','smaller','equal',
                 'name','names','date','dates','id','type','types',
                 'started','ended','released','called','located',
                 'higher','lower','different','same','both','either'}
    latin_words = re.findall(r'\b([A-Z][a-z]{2,})\b', combined)
    proper_nouns = [w for w in latin_words if w.lower() not in sql_words]
    if len(proper_nouns) >= 1:
        return True

    # Check for Arabic entity markers — names, places often have "ال" prefix
    # followed by specific patterns, but this is hard to detect reliably.
    # Instead, check if the question contains transliterated foreign words
    # (katakana-like patterns in Arabic script for names like "بيلي كوبام")
    # Simple heuristic: if there are consecutive Arabic words that don't
    # appear in a basic Arabic SQL vocabulary, likely an entity
    arabic_sql_vocab = {
        'ما', 'هو', 'هي', 'كم', 'أي', 'من', 'في', 'على', 'إلى', 'عن',
        'مع', 'بين', 'أو', 'و', 'لا', 'كل', 'جميع', 'عدد', 'اسم',
        'أسماء', 'أظهر', 'اعرض', 'جد', 'أعط', 'أكبر', 'أصغر', 'أعلى',
        'أدنى', 'أقل', 'أكثر', 'متوسط', 'مجموع', 'الذي', 'التي',
        'الذين', 'اللذين', 'كانت', 'كان', 'يكون', 'تكون', 'لديه',
        'لديها', 'يوجد', 'توجد', 'عدد', 'رقم', 'تاريخ', 'وقت',
    }

    # If numeric-only WHERE is likely (question has numbers but no text entities)
    has_numbers = bool(re.search(r'\b\d+\b', combined))
    has_comparison = bool(re.search(
        r'(أعلى|أكبر|أقل|أدنى|أكثر|greater|less|more|than|higher|lower|above|below|between|>|<|>=|<=)',
        combined, re.I))

    # Pure numeric comparison with no proper nouns → skip
    if has_numbers and has_comparison and not proper_nouns:
        return False

    # Very short questions that are just counts → skip
    count_patterns = ['كم عدد', 'how many', 'count', 'عدد']
    q_lower = combined.lower()
    if any(p in q_lower for p in count_patterns):
        # Count query — only inject if there's a named entity
        if not proper_nouns and not re.search(r'["\'"].+?["\'"]', combined):
            return False

    # Default: inject (safer to include than exclude)
    return True

def format_tier1_prompt(sample, use_ddl_override: bool = False) -> str:
    """Build enhanced prompt with all features (no dropout). Used at inference time."""
    if ENABLE_MSCHEMA and not use_ddl_override:
        schema_str = get_mschema(sample.db_id, sample.schema_ddl)
    else:
        schema_str = sample.schema_ddl[:4500]

    table_summary = build_table_column_summary(sample.schema_ddl)

    value_hints = ""
    if ENABLE_VALUE_HINTS and sample.db_path:
        value_hints = sample_db_values_question_aware(
            sample.db_path, sample.schema_ddl,
            sample.arabic_question, max_per_col=VALUE_HINT_MAX_PER_COL)

    column_hints = ""
    if ENABLE_COLUMN_LINKING:
        column_hints = link_columns_to_question(
            sample.db_id, sample.schema_ddl,
            sample.arabic_question,
            db_path=sample.db_path,
            top_k=COL_LINK_TOP_K)

    # V5 fix: skip value injection for non-entity questions (numeric/count queries)
    matched_values = ""
    if ENABLE_VALUE_INJECTION and sample.db_path:
        if _question_has_text_entities(sample.arabic_question, sample.english_question):
            matched_values = find_question_matched_values(
                sample.db_id, sample.db_path, sample.schema_ddl,
                sample.arabic_question, sample.english_question,
                top_k=VALUE_INJECTION_TOP_K, threshold=VALUE_INJECTION_THRESHOLD)

    sample_rows = ""
    if ENABLE_SAMPLE_ROWS and sample.db_path:
        sample_rows = get_sample_rows(
            sample.db_path, sample.schema_ddl, column_hints,
            limit=SAMPLE_ROWS_LIMIT, max_tables=SAMPLE_ROWS_MAX_TABLES)

    # FIX 4: Always include English hint (no dropout)
    english_hint = ""
    if sample.english_question:
        english_hint = f"(English: {sample.english_question})\n"

    return TIER1_PROMPT.format(
        schema=schema_str, value_hints=value_hints,
        matched_values=matched_values,
        column_hints=column_hints,
        sample_rows=sample_rows,
        table_summary=table_summary, question=sample.arabic_question,
        english_hint=english_hint)


def prewarm_column_cache():
    """Pre-compute column embeddings for all databases."""
    if _emb_model is None:
        print("⚠️ Embedding model not loaded — skipping column cache")
        return
    count = 0
    for db_id, schema_ddl in ar_spider_schemas.items():
        _get_column_embeddings(db_id, schema_ddl)
        count += 1
    total_cols = sum(len(v['candidates']) for v in _col_emb_cache.values())
    print(f"✅ Column embeddings cached for {count} databases ({total_cols} columns)")


def prewarm_value_cache():
    """Pre-compute value embeddings for all databases with SQLite files."""
    if _emb_model is None:
        print("⚠️ Embedding model not loaded — skipping value cache")
        return
    count = 0
    total_vals = 0
    for db_id, schema_ddl in ar_spider_schemas.items():
        db_path = db_paths.get(db_id, "")
        if db_path:
            data = _build_value_embeddings(db_id, db_path, schema_ddl)
            total_vals += len(data["values"])
            count += 1
    print(f"✅ Value embeddings cached for {count} databases ({total_vals} unique values)")


print("✅ Shared functions defined (M-Schema, column linking, value hints, "
      "sample rows, prompt builders)")

## 📂 Load dev.json → Evaluation Data

In [ ]:
#@title 📂 Load dev.json → Evaluation Data { display-mode: "form" }

print("📂 Loading Ar-Spider dev.json...")
with open(os.path.join(AR_SPIDER_DIR, "dev.json"), "r", encoding="utf-8") as f:
    dev_raw = json.load(f)
print(f"   Raw samples: {len(dev_raw)}")

@dataclass
class EvalSample:
    id: str
    db_id: str
    arabic_question: str
    gold_sql: str
    schema_ddl: str
    db_path: str = ""
    english_question: str = ""

eval_samples = []
for idx, r in enumerate(dev_raw):
    db_id = r["db_id"]
    schema = ar_spider_schemas.get(db_id, "")
    if not schema:
        continue
    eval_samples.append(EvalSample(
        id=f"ar_{idx:04d}",
        db_id=db_id,
        arabic_question=r.get("Arabic", r.get("question", "")),
        gold_sql=r.get("query", ""),
        schema_ddl=schema,
        db_path=db_paths.get(db_id, ""),
        english_question=r.get("question", ""),
    ))

print(f"✅ Evaluation set: {len(eval_samples)} samples (full dev)")
print(f"   Unique databases: {len(set(s.db_id for s in eval_samples))}")
eng_count = sum(1 for s in eval_samples if s.english_question)
print(f"   English questions: {eng_count}/{len(eval_samples)} ({'✅' if eng_count == len(eval_samples) else '⚠️'})")
if eval_samples:
    s = eval_samples[0]
    print(f"   Sample: AR: {s.arabic_question[:60]}")
    print(f"           EN: {s.english_question[:60]}")
print(f"\n   ⚠️ ZERO overlap with training data (train.json ≠ dev.json)")

---
## 🔄 Load Adapter from Google Drive (Inference)

Reconnect runtime after training, then load the saved adapter.

In [ ]:
#@title 🔄 Load Adapter from Google Drive { display-mode: "form" }

from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

DRIVE_SAVE_DIR = "/content/drive/MyDrive/arabic_text2sql_7b_adapter_v5_aligned"  #@param {type:"string"}
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

# Load adapter from Drive
infer_model = PeftModel.from_pretrained(base_model, DRIVE_SAVE_DIR)
infer_model.eval()

# Load tokenizer
infer_tok = AutoTokenizer.from_pretrained(DRIVE_SAVE_DIR)
if infer_tok.pad_token is None:
    infer_tok.pad_token = infer_tok.eos_token

print(f"✅ Model loaded: {BASE_MODEL} + adapter from Google Drive")
print(f"   Adapter: {DRIVE_SAVE_DIR}")

## 🔄 Load Embedding Model for Column Linking (Inference)

In [ ]:
#@title 🔄 Load Embedding Model for Column Linking { display-mode: "form" }

from sentence_transformers import SentenceTransformer
import numpy as np

print("🔄 Loading BGE-M3 embedding model for column linking...")
_emb_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cuda",
)
print(f"✅ BGE-M3 loaded ({_emb_model.get_sentence_embedding_dimension()}d, ~2.3GB VRAM)")

# Pre-warm caches for inference
print("🔄 Pre-warming embedding caches...")
_col_emb_cache.clear()
_db_value_emb_cache.clear()
prewarm_column_cache()
prewarm_value_cache()
print("✅ Embedding caches ready for inference")

## 🇸🇦 Load Arabic Descriptions + Table Glosses (Inference)

In [ ]:
#@title 🇸🇦 Load Arabic Descriptions + Table Glosses { display-mode: "form" }

# Load Arabic column descriptions from cache
if ENABLE_AR_COL_DESC and os.path.exists(AR_COL_DESC_PATH):
    with open(AR_COL_DESC_PATH, "r", encoding="utf-8") as f:
        _ar_col_descriptions = json.load(f)
    total_descs = sum(len(v) for v in _ar_col_descriptions.values())
    print(f"✅ Loaded {total_descs} Arabic column descriptions from cache")
else:
    print(f"⚠️ Arabic column descriptions not found \u2014 will generate using model")
    # You can regenerate them here if needed (see V4 cells in original notebook)

# Load Arabic table glosses from cache
if ENABLE_BILINGUAL_SCHEMA and os.path.exists(AR_TABLE_GLOSS_PATH):
    with open(AR_TABLE_GLOSS_PATH, "r", encoding="utf-8") as f:
        _ar_table_glosses = json.load(f)
    total_glosses = sum(len(v) for v in _ar_table_glosses.values())
    print(f"✅ Loaded {total_glosses} Arabic table glosses from cache")
else:
    print(f"⚠️ Table glosses not found \u2014 M-Schema will use English-only names")

# Rebuild column embeddings with bilingual text
if ENABLE_BILINGUAL_EMBEDDINGS and _ar_col_descriptions and ENABLE_COLUMN_LINKING:
    print("🔄 Rebuilding column embeddings with bilingual (EN+AR) descriptions...")
    _col_emb_cache.clear()
    prewarm_column_cache()
    print("✅ Column embeddings now include Arabic descriptions")

---
## 🌌 Translate Dev Questions to English (Google Translate)

Translate Arabic dev questions to English for the `(English: ...)` inference hint. Cached to Drive.

In [ ]:
"""## 🌐 V4: English Translation Hint (Google Translate)"""

# ═══════════════════════════════════════════════════════════════════
# V4 TECHNIQUE 5c: English Translation Hint (Google Translate + DB Grounding)
# ═══════════════════════════════════════════════════════════════════
# Two-phase English hint generation (generalizable to ANY language):
#
#   Phase 1 — Translate Arabic → English using Google Translate
#     Best-in-class Arabic translation, no VRAM cost, no model loading.
#     Uses deep-translator library (free, no API key needed).
#
#   Phase 2 — Ground values against actual database
#     Fuzzy-match noun phrases in the translation against real
#     DB values, replace near-matches with exact values:
#     "Middle Africa" → "Central Africa"
#
# Cached to Drive — first run ~2-3 min, subsequent runs instant.
# Research: Cross-lingual translation with database-grounded value alignment.
# ═══════════════════════════════════════════════════════════════════

ENGLISH_TRANS_CACHE = "/content/drive/MyDrive/ar_spider_english_grounded.json"  #@param {type:"string"}
REGENERATE_TRANSLATIONS = False  #@param {type:"boolean"}
GROUNDING_SIM_THRESHOLD = 0.65  #@param {type:"number"}

from difflib import SequenceMatcher as _SM

# ── Load Google Translate ────────────────────────────────────────

_translator = None

def _load_translator():
    global _translator
    if _translator is not None:
        return
    try:
        from deep_translator import GoogleTranslator
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "deep-translator"], check=True)
        from deep_translator import GoogleTranslator
    _translator = GoogleTranslator(source='ar', target='en')
    print("✅ Google Translate loaded (no VRAM cost)")


# ── Phase 1: Google Translate ────────────────────────────────────

def _translate_arabic_to_english(arabic_text: str) -> str:
    """Translate Arabic → English using Google Translate."""
    try:
        result = _translator.translate(arabic_text)
        return result.strip() if result and len(result.strip()) > 3 else ""
    except Exception as e:
        # Rate limit or network error — retry once after short delay
        import time
        time.sleep(0.5)
        try:
            result = _translator.translate(arabic_text)
            return result.strip() if result and len(result.strip()) > 3 else ""
        except Exception:
            return ""


# ── Phase 2: DB Value Grounding ──────────────────────────────────

def _load_db_text_values(db_path: str, schema_ddl: str) -> list:
    """Load all distinct TEXT values from a database."""
    if not db_path or not os.path.exists(db_path):
        return []
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return []
    all_values = set()
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl, cols in schema_tables.items():
            for col in cols:
                try:
                    cursor.execute(f'PRAGMA table_info("{tbl}")')
                    col_info = {row[1].lower(): row[2] for row in cursor.fetchall()}
                    col_type = col_info.get(col.lower(), "TEXT").upper()
                    if not any(t in col_type for t in ["TEXT", "VARCHAR", "CHAR", "CLOB"]):
                        continue
                    cursor.execute(
                        f'SELECT DISTINCT "{col}" FROM "{tbl}" '
                        f'WHERE "{col}" IS NOT NULL AND TRIM("{col}") != "" '
                        f'LIMIT 100')
                    for row in cursor.fetchall():
                        val = str(row[0]).strip()
                        if len(val) >= 2:
                            all_values.add(val)
                except Exception:
                    continue
        conn.close()
    except Exception:
        pass
    return list(all_values)

_db_value_cache = {}

def _get_db_values(db_id: str, db_path: str, schema_ddl: str) -> list:
    """Get cached DB values for a database."""
    if db_id not in _db_value_cache:
        _db_value_cache[db_id] = _load_db_text_values(db_path, schema_ddl)
    return _db_value_cache[db_id]


def ground_translation(translation: str, db_values: list,
                        threshold: float = 0.65) -> tuple:
    """
    Ground a translated English question against actual DB values.
    Returns: (grounded_translation, list_of_replacements)
    """
    if not translation or not db_values:
        return translation, []

    replacements = []
    MIN_VALUE_LEN = 4
    safe_values = [v for v in db_values if len(v) >= MIN_VALUE_LEN]
    db_vals_lower = {v.lower(): v for v in safe_values}
    grounded = translation

    # Step 1: Exact case-insensitive match (word boundaries)
    for db_val in sorted(safe_values, key=len, reverse=True):
        try:
            pattern = re.compile(r'\b' + re.escape(db_val) + r'\b', re.IGNORECASE)
            match = pattern.search(grounded)
            if match and match.group(0) != db_val:
                grounded = pattern.sub(db_val, grounded)
                replacements.append(f"case: '{match.group(0)}' → '{db_val}'")
        except re.error:
            continue

    # Step 2: Extract candidate phrases for fuzzy matching
    candidates = []
    for m in re.finditer(r'"([^"]+)"', grounded):
        candidates.append((m.start(), m.end(), m.group(1), True))
    for m in re.finditer(r"'([^']+)'", grounded):
        candidates.append((m.start(), m.end(), m.group(1), True))
    for m in re.finditer(
        r'(?:[A-Z][a-z]+(?:\s+(?:of|the|and|in|for|to|de|la|le|el|al))?'
        r'(?:\s+[A-Z][a-z]+)*(?:\s+\([^)]+\))?)', grounded):
        phrase = m.group(0).strip()
        if len(phrase) >= MIN_VALUE_LEN:
            already_covered = any(
                s <= m.start() and m.end() <= e
                for s, e, _, q in candidates if q)
            if not already_covered:
                candidates.append((m.start(), m.end(), phrase, False))

    # Step 3: Fuzzy match candidates against DB values
    candidates.sort(key=lambda x: -x[0])
    skip_words = {
        'the', 'what', 'which', 'where', 'when', 'who', 'how',
        'find', 'show', 'list', 'give', 'get', 'display',
        'count', 'number', 'average', 'maximum', 'minimum',
        'total', 'all', 'each', 'every', 'many', 'most',
        'name', 'type', 'have', 'has', 'had', 'does', 'did',
        'with', 'from', 'that', 'this', 'than', 'then',
        'more', 'less', 'much', 'some', 'any', 'not',
        'and', 'but', 'for', 'are', 'were', 'been', 'being',
    }
    for start, end, phrase, is_quoted in candidates:
        phrase_lower = phrase.lower()
        if phrase_lower in skip_words:
            continue
        if len(phrase) < MIN_VALUE_LEN:
            continue
        if phrase in safe_values or phrase_lower in db_vals_lower:
            continue
        best_score = 0
        best_match = None
        for db_val in safe_values:
            len_ratio = len(phrase) / max(len(db_val), 1)
            if len_ratio < 0.4 or len_ratio > 2.5:
                continue
            score = _SM(None, phrase_lower, db_val.lower()).ratio()
            if score > best_score:
                best_score = score
                best_match = db_val
        if best_match and best_score >= threshold:
            before_char = grounded[start - 1] if start > 0 else ' '
            after_char = grounded[end] if end < len(grounded) else ' '
            if before_char.isalnum() or after_char.isalnum():
                continue
            grounded = grounded[:start] + best_match + grounded[end:]
            replacements.append(f"fuzzy({best_score:.2f}): '{phrase}' → '{best_match}'")

    return grounded, replacements


# ── Combined Pipeline: Translate + Ground + Cache ────────────────

def generate_grounded_translations():
    """Generate English translations with DB value grounding for all eval samples."""
    cache_exists = os.path.exists(ENGLISH_TRANS_CACHE)

    # Try cache
    if cache_exists and not REGENERATE_TRANSLATIONS:
        try:
            with open(ENGLISH_TRANS_CACHE, "r", encoding="utf-8") as f:
                cached = json.load(f)
            print(f"📂 Found translation cache: {len(cached)} entries")
            matched = 0
            missing = []
            for s in eval_samples:
                entry = cached.get(s.id, {})
                if isinstance(entry, dict):
                    eng = entry.get("raw", "")
                elif isinstance(entry, str):
                    eng = entry
                else:
                    eng = ""
                if eng:
                    s.english_question = eng
                    matched += 1
                else:
                    missing.append(s)
            print(f"   Matched: {matched}/{len(eval_samples)}")

            if missing:
                print(f"   ⚠️ {len(missing)} missing — translating + grounding...")
                _load_translator()
                for i, s in enumerate(missing):
                    raw = _translate_arabic_to_english(s.arabic_question)
                    if raw:
                        db_vals = _get_db_values(s.db_id, s.db_path, s.schema_ddl)
                        grounded, reps = ground_translation(raw, db_vals, GROUNDING_SIM_THRESHOLD)
                        s.english_question = raw  # use raw Google Translate (grounding saved for analysis only)
                        cached[s.id] = {"raw": raw, "grounded": grounded, "replacements": reps}
                        matched += 1
                    if (i + 1) % 50 == 0:
                        print(f"      [{i+1}/{len(missing)}]")
                with open(ENGLISH_TRANS_CACHE, "w", encoding="utf-8") as f:
                    json.dump(cached, f, ensure_ascii=False, indent=2)
                print(f"   ✅ Updated cache: {len(cached)} total")
            return
        except (json.JSONDecodeError, Exception) as e:
            print(f"⚠️ Cache failed: {e}")

    # Full generation
    _load_translator()
    if cache_exists and REGENERATE_TRANSLATIONS:
        print(f"🔄 REGENERATE_TRANSLATIONS=True — regenerating all")
    else:
        print(f"📂 No translation cache found")
    print(f"🌐 Translating + grounding {len(eval_samples)} samples...")
    print(f"   Translation: Google Translate (deep-translator)")
    print(f"   Grounding threshold: {GROUNDING_SIM_THRESHOLD}")
    print()

    cached = {}
    total_replacements = 0
    grounded_count = 0

    for i, s in enumerate(eval_samples):
        raw = _translate_arabic_to_english(s.arabic_question)
        if raw:
            db_vals = _get_db_values(s.db_id, s.db_path, s.schema_ddl)
            grounded, reps = ground_translation(raw, db_vals, GROUNDING_SIM_THRESHOLD)
            s.english_question = raw  # use raw Google Translate (grounding saved for analysis only)
            cached[s.id] = {"raw": raw, "grounded": grounded, "replacements": reps}
            if reps:
                grounded_count += 1
                total_replacements += len(reps)
        if (i + 1) % 100 == 0 or i == len(eval_samples) - 1:
            print(f"   [{i+1}/{len(eval_samples)}] translated: {len(cached)}, "
                  f"grounded: {grounded_count} ({total_replacements} replacements)")

    os.makedirs(os.path.dirname(ENGLISH_TRANS_CACHE), exist_ok=True)
    with open(ENGLISH_TRANS_CACHE, "w", encoding="utf-8") as f:
        json.dump(cached, f, ensure_ascii=False, indent=2)

    populated = sum(1 for s in eval_samples if s.english_question)
    print(f"\n✅ Done: {len(cached)} translations → {ENGLISH_TRANS_CACHE}")
    print(f"   Grounded: {grounded_count} samples ({total_replacements} value replacements)")
    print(f"   Populated: {populated}/{len(eval_samples)} eval samples")

    # Show examples
    print(f"\n📋 Grounding examples:")
    shown = 0
    for sid, entry in cached.items():
        if isinstance(entry, dict) and entry.get("replacements") and shown < 8:
            print(f"\n   {sid}:")
            print(f"     Raw:      {entry['raw'][:80]}")
            print(f"     Grounded: {entry['grounded'][:80]}")
            for rep in entry["replacements"]:
                print(f"     Fix: {rep}")
            shown += 1
    if shown < 3:
        print(f"\n📋 Translation examples:")
        plain_shown = 0
        for s in eval_samples[:20]:
            if s.english_question and plain_shown < 5:
                print(f"   {s.id}: AR: {s.arabic_question[:60]}")
                print(f"         EN: {s.english_question[:60]}")
                plain_shown += 1


if ENABLE_ENGLISH_HINT:
    generate_grounded_translations()
else:
    print("ℹ️  English hints disabled (ENABLE_ENGLISH_HINT = False)")

---
## 🔧 Inference Post-Processing Functions

SQL extraction, identifier fixing, hallucinated filter removal, SQL value grounding.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DEPENDENCY FUNCTIONS
# ═══════════════════════════════════════════════════════════════════
# These are copied from earlier cells so this cell is self-contained.
# If you've already run the earlier cells, these are harmless re-defs.
# ═══════════════════════════════════════════════════════════════════

from difflib import SequenceMatcher


def extract_sql_from_response(response: str) -> str:
    """Extract clean SQL from model response."""
    m = re.search(r'```(?:sql)?\s*\n?(.*?)```', response, re.DOTALL | re.I)
    if m:
        sql = m.group(1).strip()
        lines = [l for l in sql.split("\n") if not l.strip().startswith("--")]
        return "\n".join(lines).strip().rstrip(";")
    m = re.search(r'((?:SELECT|WITH)\s+.+?\bFROM\b.+?)(?:;|\n\n|$)', response, re.I | re.DOTALL)
    if m:
        return m.group(1).strip().rstrip(";")
    return response.strip().rstrip(";")


def extract_tables_columns(schema_sql: str) -> Dict[str, List[str]]:
    """Parse CREATE TABLE DDL into {table_name: [column_names]}."""
    tables = {}
    cur_table = None
    for line in schema_sql.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I,
        )
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+\w+', line)
            if cm and not any(
                line.upper().startswith(kw) for kw in
                ('PRIMARY', 'FOREIGN', 'UNIQUE', 'CHECK', 'CONSTRAINT', '--')
            ):
                tables[cur_table].append(cm.group(1))
    return tables


def normalize_for_match(name: str) -> str:
    return name.lower().replace("_", "")


def find_best_match(identifier: str, candidates: List[str], threshold: float = 0.7) -> Optional[str]:
    id_norm = normalize_for_match(identifier)
    for c in candidates:
        if c.lower() == identifier.lower():
            return c
    for c in candidates:
        if normalize_for_match(c) == id_norm:
            return c
    best_score, best_match = 0, None
    for c in candidates:
        score = SequenceMatcher(None, id_norm, normalize_for_match(c)).ratio()
        if score > best_score:
            best_score, best_match = score, c
    return best_match if best_score >= threshold else None


SQL_KEYWORDS = {
    'select','from','where','join','inner','left','right','outer','cross',
    'natural','using','on','and','or','not','in','exists','between','like',
    'glob','is','null','as','escape',
    'order','by','group','having','limit','offset','union','intersect',
    'except','all','distinct','case','when','then','else','end','cast',
    'asc','desc','values','insert','update','delete','set','into',
    'top','iif','no',
    'count','sum','avg','min','max','abs','length','substr','substring',
    'replace','trim','ltrim','rtrim','round','upper','lower','instr',
    'coalesce','ifnull','nullif','typeof','total','group_concat',
    'create','table','primary','key','foreign','references','unique',
    'check','constraint','default','autoincrement','index','collate',
    'integer','text','real','boolean','date','time','varchar',
    'char','float','double','blob','numeric','int',
    'true','false',
    'with','recursive','over','partition','row','rows','fetch','next',
    'first','last','only','nulls','current','preceding','following',
    'unbounded','range','groups','window','filter',
    'rank','dense_rank','row_number','ntile','lag','lead',
    'first_value','last_value','nth_value',
}


def fix_sql_identifiers(pred_sql: str, schema_ddl: str) -> Tuple[str, List[str]]:
    """Fix column/table names in predicted SQL by matching against schema."""
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return pred_sql, []

    all_table_names = list(schema_tables.keys())
    all_column_names = []
    for tbl, cols in schema_tables.items():
        all_column_names.extend(cols)

    seen_lower = set()
    unique_columns = []
    for c in all_column_names:
        if c.lower() not in seen_lower:
            seen_lower.add(c.lower())
            unique_columns.append(c)

    fixes = []
    fixed = pred_sql

    # Pass 1: Fix table names after FROM/JOIN
    table_pat = re.compile(r'(?:FROM|JOIN)\s+([A-Za-z_]\w*)', re.I)
    for m in table_pat.finditer(fixed):
        used = m.group(1)
        if used.lower() in SQL_KEYWORDS:
            continue
        match = find_best_match(used, all_table_names, threshold=0.6)
        if match and match != used:
            fixed = re.sub(r'\b' + re.escape(used) + r'\b', match, fixed)
            fixes.append(f"table: {used} → {match}")

    # Pass 2: Fix qualified columns (T1.col, table.col)
    qual_pat = re.compile(r'(\b[A-Za-z_]\w*\b)\.(\b[A-Za-z_]\w*\b)')
    for m in qual_pat.finditer(fixed):
        col_ref = m.group(2)
        if col_ref.lower() in SQL_KEYWORDS:
            continue
        match = find_best_match(col_ref, unique_columns, threshold=0.65)
        if match and match != col_ref:
            old = f"{m.group(1)}.{col_ref}"
            new = f"{m.group(1)}.{match}"
            fixed = fixed.replace(old, new)
            fixes.append(f"col: {old} → {new}")

    # Pass 3: Fix unqualified columns via variant generation
    for col in unique_columns:
        variants = set()
        snake = re.sub(r'(?<=[a-z])([A-Z])', r'_\1', col).lower()
        if snake != col.lower():
            variants.add(snake)
        parts = col.split('_')
        if len(parts) > 1:
            camel = parts[0].lower() + ''.join(p.capitalize() for p in parts[1:])
            variants.add(camel)
            variants.add(camel.lower())
            pascal = ''.join(p.capitalize() for p in parts)
            variants.add(pascal)
        variants.add(col.lower())
        variants.add(col.upper())
        variants.discard(col)

        for variant in variants:
            if variant and variant != col:
                pat = r'\b' + re.escape(variant) + r'\b'
                if re.search(pat, fixed):
                    fixed = re.sub(pat, col, fixed)
                    fixes.append(f"variant: {variant} → {col}")

    return fixed, fixes


# Arabic ↔ English value grounding map
VALUE_GROUNDING_MAP = {
    "dog":      ["كلب", "كلاب", "الكلب", "الكلاب", "كلبا"],
    "cat":      ["قط", "قطة", "قطط", "القط", "القطة", "القطط", "هر", "هرة"],
    "bird":     ["طائر", "طيور", "الطائر", "الطيور", "عصفور"],
    "rabbit":   ["أرنب", "أرانب", "الأرنب"],
    "elephant": ["فيل", "أفيال", "الفيل"],
    "parrot":   ["ببغاء", "الببغاء"],
    "f":        ["طالبات", "الطالبات", "أنثى", "إناث", "نساء", "امرأة", "بنات", "البنات", "طالبة"],
    "m":        ["طلاب", "الطلاب", "ذكر", "ذكور", "رجال", "رجل", "أولاد"],
    "female":   ["طالبات", "الطالبات", "أنثى", "إناث", "نساء", "امرأة"],
    "male":     ["طلاب", "ذكر", "ذكور", "رجال"],
    "france":   ["فرنسا", "الفرنسية", "فرنسي"],
    "usa":      ["الولايات المتحدة", "أمريكا", "أمريكي", "الأمريكية"],
    "japan":    ["اليابان", "ياباني", "اليابانية"],
    "germany":  ["ألمانيا", "ألماني", "الألمانية"],
    "europe":   ["أوروبا", "أوروبي", "الأوروبية"],
    "asia":     ["آسيا", "آسيوي", "الآسيوية"],
    "africa":   ["أفريقيا", "أفريقي", "الأفريقية"],
    "rock":     ["روك", "الروك"],
    "pop":      ["بوب", "البوب"],
    "jazz":     ["جاز", "الجاز"],
    "classical":["كلاسيكي", "الكلاسيكية"],
    "yes":      ["نعم"],
    "no":       ["لا"],
    "true":     ["صحيح", "نعم"],
    "false":    ["خطأ", "لا"],
}


def is_value_grounded(value: str, question: str) -> bool:
    """Check if a string literal value from SQL is grounded in the Arabic question."""
    val_lower = value.strip().lower()
    question_lower = question.lower()
    if val_lower in question_lower:
        return True
    if val_lower in VALUE_GROUNDING_MAP:
        arabic_words = VALUE_GROUNDING_MAP[val_lower]
        for aw in arabic_words:
            if aw in question:
                return True
        return False
    try:
        float(val_lower)
        return True
    except ValueError:
        pass
    if len(value.split()) > 1:
        return True
    if val_lower not in VALUE_GROUNDING_MAP:
        return True
    return False


def remove_hallucinated_filters(sql: str, question: str) -> Tuple[str, List[str]]:
    """Remove WHERE conditions with string literals not grounded in the Arabic question."""
    removals = []
    where_match = re.search(
        r'\bWHERE\b\s+(.*?)(?:\bGROUP\b|\bORDER\b|\bLIMIT\b|\bHAVING\b|\bUNION\b|\bINTERSECT\b|\bEXCEPT\b|$)',
        sql, re.I | re.DOTALL
    )
    if not where_match:
        return sql, []

    where_clause = where_match.group(1).strip()
    where_start = where_match.start(1)
    where_end = where_match.end(1)

    conditions = []
    depth = 0
    current = ""
    tokens = re.split(r'(\bAND\b)', where_clause, flags=re.I)
    i = 0
    while i < len(tokens):
        token = tokens[i]
        depth += token.count('(') - token.count(')')
        if token.strip().upper() == 'AND' and depth == 0:
            if current.strip():
                conditions.append(current.strip())
            current = ""
        else:
            current += token
        i += 1
    if current.strip():
        conditions.append(current.strip())

    keep_conditions = []
    for cond in conditions:
        literals = re.findall(r"""["'](.*?)["']""", cond)
        if not literals:
            keep_conditions.append(cond)
            continue
        all_grounded = True
        for lit in literals:
            if not is_value_grounded(lit, question):
                all_grounded = False
                break
        if all_grounded:
            keep_conditions.append(cond)
        else:
            removals.append(f"removed: {cond.strip()[:80]}")

    final_conditions = []
    for cond in keep_conditions:
        sub_literals = re.findall(r"""["'](.*?)["']""", cond)
        sub_hallucinated = False
        for lit in sub_literals:
            if not is_value_grounded(lit, question):
                sub_hallucinated = True
                break
        if sub_hallucinated:
            removals.append(f"removed (subquery): {cond.strip()[:80]}")
        else:
            final_conditions.append(cond)

    if not removals:
        return sql, []

    if final_conditions:
        new_where = " AND ".join(final_conditions)
        new_sql = sql[:where_start] + new_where + sql[where_end:]
    else:
        where_full = re.search(
            r'\bWHERE\b\s+.*?(?=\bGROUP\b|\bORDER\b|\bLIMIT\b|\bHAVING\b|\bUNION\b|\bINTERSECT\b|\bEXCEPT\b|$)',
            sql, re.I | re.DOTALL
        )
        if where_full:
            new_sql = sql[:where_full.start()].rstrip() + " " + sql[where_full.end():].lstrip()
        else:
            new_sql = sql
    new_sql = re.sub(r'\s+', ' ', new_sql).strip()
    return new_sql, removals


print("✅ Dependency functions loaded (extract_sql, fix_identifiers, filter_removal)")


# ── SQL Value Grounding Post-Processor ───────────────────────────
# After SQL generation, check every string literal against actual
# DB values. If a literal doesn't exist but a close match does,
# replace it. Fixes: 'Middle Africa' → 'Central Africa',
# 'general motors' → 'General Motors', etc.
# Uses cosine similarity via the loaded embedding model (_emb_model)
# for semantic matching instead of string-based fuzzy matching.

def ground_sql_values(sql: str, db_path: str, schema_ddl: str) -> Tuple[str, List[str]]:
    """
    Replace string literals in SQL with exact DB values using cosine similarity.
    Uses the already-loaded embedding model (_emb_model) for semantic matching.
    Returns: (fixed_sql, list_of_replacements)
    """
    if not sql or not db_path or not os.path.exists(db_path):
        return sql, []

    # Extract string literals from SQL
    literals = []
    for m in re.finditer(r"'([^']*)'", sql):
        val = m.group(1)
        if val and len(val) >= 2:
            literals.append((m.start(1), m.end(1), val))

    if not literals:
        return sql, []

    def _find_column_for_literal(sql_text, lit_start):
        """Look backwards from literal position to find column name."""
        prefix = sql_text[:lit_start].strip()
        m = re.search(r'(\w+(?:\.\w+)?)\s*(?:=|!=|<>|LIKE|like|IN|in)\s*["\']?\s*$', prefix)
        return m.group(1) if m else None

    # Load DB values per column
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return sql, []

    col_values = {}
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl, cols in schema_tables.items():
            for col in cols:
                try:
                    cursor.execute(f'PRAGMA table_info("{tbl}")')
                    col_info = {row[1].lower(): row[2] for row in cursor.fetchall()}
                    col_type = col_info.get(col.lower(), "TEXT").upper()
                    if not any(t in col_type for t in ["TEXT", "VARCHAR", "CHAR", "CLOB"]):
                        continue
                    cursor.execute(
                        f'SELECT DISTINCT "{col}" FROM "{tbl}" '
                        f'WHERE "{col}" IS NOT NULL AND TRIM("{col}") != "" '
                        f'LIMIT 200')
                    vals = set()
                    for row in cursor.fetchall():
                        v = str(row[0]).strip()
                        if v:
                            vals.add(v)
                    if vals:
                        col_values[f"{tbl}.{col}".lower()] = vals
                        col_values[col.lower()] = vals
                except Exception:
                    continue
        conn.close()
    except Exception:
        return sql, []

    if not col_values:
        return sql, []

    all_values = set()
    for vals in col_values.values():
        all_values.update(vals)

    # Check each literal and replace if needed (right-to-left)
    replacements = []
    fixed_sql = sql

    for start, end, literal in sorted(literals, key=lambda x: -x[0]):
        if literal in all_values:
            continue  # exact match — no fix needed

        # Case-insensitive exact match first (free, no embedding needed)
        lit_lower = literal.lower()
        case_match = None
        for v in all_values:
            if v.lower() == lit_lower:
                case_match = v
                break
        if case_match:
            fixed_sql = fixed_sql[:start] + case_match + fixed_sql[end:]
            replacements.append(f"case: '{literal}' → '{case_match}'")
            continue

        # Find target column for this literal
        col_name = _find_column_for_literal(fixed_sql, start - 1)
        if col_name:
            candidates = col_values.get(col_name.lower(), set())
            if not candidates:
                bare = col_name.split(".")[-1].lower()
                candidates = col_values.get(bare, set())
        else:
            candidates = all_values

        if not candidates:
            continue

        # ── Cosine similarity matching using embedding model ─────
        candidates_list = [v for v in candidates if len(v) >= 2]
        if not candidates_list:
            continue

        try:
            # Encode the literal and all candidate DB values
            lit_embedding = _emb_model.encode(
                [literal], normalize_embeddings=True,
                show_progress_bar=False,
            )[0]
            cand_embeddings = _emb_model.encode(
                candidates_list, normalize_embeddings=True,
                show_progress_bar=False,
            )

            # Cosine similarity (dot product of normalized vectors)
            similarities = cand_embeddings @ lit_embedding
            best_idx = int(np.argmax(similarities))
            best_score = float(similarities[best_idx])
            best_match = candidates_list[best_idx]

            if best_score >= SQL_VALUE_SIM_THRESHOLD:
                fixed_sql = fixed_sql[:start] + best_match + fixed_sql[end:]
                replacements.append(
                    f"cosine({best_score:.2f}): '{literal}' → '{best_match}' (col={col_name or '?'})")
        except Exception:
            continue

    return fixed_sql, replacements

---
## 👁️ Preview: What the Inference Model Sees

In [ ]:
#@title 👁️ Preview: What the Model Sees { display-mode: "form" }

# ═══════════════════════════════════════════════════════════════════
# Shows the FULL prompt for a few eval samples so you can verify
# that column hints + Arabic descriptions are present and correct.
# ═══════════════════════════════════════════════════════════════════

PREVIEW_SAMPLE_IDS = [0, 25, 50, 100, 200, 400, 600, 800,  1000]  #@param {type:"raw"}

print("=" * 80)
print("  👁️ PROMPT PREVIEW — What the model receives at inference")
print("=" * 80)

for sample_idx in PREVIEW_SAMPLE_IDS:
    if sample_idx >= len(eval_samples):
        continue
    sample = eval_samples[sample_idx]

    prompt = format_tier1_prompt(sample)

    # Count tokens (rough estimate)
    try:
        token_count = len(infer_tok.encode(prompt))
    except:
        token_count = len(prompt) // 3  # rough fallback

    print(f"\n{'━' * 80}")
    print(f"  Sample: {sample.id} | {sample.db_id} | ~{token_count} tokens")
    print(f"  Question: {sample.arabic_question[:100]}")
    print(f"  Gold SQL: {sample.gold_sql[:100]}")
    print(f"{'━' * 80}")

    # Print prompt with section markers
    for line in prompt.split("\n"):
        # Highlight key V4 sections
        if line.strip().startswith("★"):
            print(f"  \033[92m{line}\033[0m")  # green for column hints
        elif line.strip().startswith("Likely relevant"):
            print(f"  \033[92m{line}\033[0m")  # green header
        elif line.strip().startswith("Sample column values"):
            print(f"  \033[93m{line}\033[0m")  # yellow for value hints
        elif line.strip().startswith("Question:"):
            print(f"  \033[96m{line}\033[0m")  # cyan for question
        else:
            print(f"  {line}")

    print()

# ── Summary stats ────────────────────────────────────────────────
print("=" * 80)
print("  📊 PROMPT STATISTICS (across all eval samples)")
print("=" * 80)

token_counts = []
col_hint_counts = []
ar_desc_counts = []

for s in eval_samples[:50]:  # sample first 50 for speed
    p = format_tier1_prompt(s)
    try:
        tc = len(infer_tok.encode(p))
    except:
        tc = len(p) // 3
    token_counts.append(tc)
    col_hint_counts.append(p.count("★"))
    ar_desc_counts.append(p.count("—"))

print(f"  Token count (first 50 samples):")
print(f"    Min: {min(token_counts)}, Max: {max(token_counts)}, Avg: {sum(token_counts)//len(token_counts)}")
print(f"    Over MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH_7B}): {sum(1 for t in token_counts if t > MAX_SEQ_LENGTH_7B)}/{len(token_counts)}")
print(f"  Column hints per prompt: avg {sum(col_hint_counts)/len(col_hint_counts):.1f}")
print(f"  Arabic descriptions per prompt: avg {sum(ar_desc_counts)/len(ar_desc_counts):.1f}")
print()

---
## ⚡ Batched Generation + Self-Consistency Voting

Generates N candidates per sample (1 greedy + multi-temperature sampled), executes each against the DB, and votes by execution result.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BATCHED GENERATION (the key optimization)
# ═══════════════════════════════════════════════════════════════════
# Instead of calling model.generate() N times sequentially,
# we batch all N candidates into a single (or few) GPU forward passes.
#
# For SC with N=8 and batch_size=4:
#   Pass 1: 1 greedy candidate
#   Pass 2: 4 sampled candidates (batch)
#   Pass 3: 3 sampled candidates (batch)
# = 3 GPU passes instead of 8 sequential ones → ~3x faster
# ═══════════════════════════════════════════════════════════════════

def _build_chat_text(tokenizer, prompt: str) -> str:
    """Build chat-formatted string from a user prompt."""
    messages = [
        {"role": "system", "content": TIER1_SYSTEM},
        {"role": "user", "content": prompt},
    ]
    return tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False)


def generate_candidates_batched(model, tokenizer, prompt: str,
                                 n_greedy: int = 1,
                                 n_sampled: int = 0,
                                 batch_size: int = 4,
                                 temperature: float = 0.7,
                                 top_p: float = 0.95) -> List[str]:
    """
    Generate multiple SQL candidates for ONE prompt in batched GPU passes.

    Args:
        n_greedy:  number of greedy decodes (typically 1)
        n_sampled: number of sampled candidates (typically N-1)
        batch_size: max sequences per GPU pass (tune to VRAM)

    Returns:
        List of raw SQL strings (greedy first, then sampled).
    """
    chat = _build_chat_text(tokenizer, prompt)
    results = []

    # Save and set padding side for batched generation
    orig_pad_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    # ── Greedy pass (1 candidate) ──────────────────────────────
    if n_greedy > 0:
        try:
            inp = tokenizer(
                [chat], return_tensors="pt",
                truncation=True, max_length=MAX_SEQ_LENGTH_7B,
            ).to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inp, max_new_tokens=300,
                    do_sample=False,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                )
            resp = tokenizer.decode(
                out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
            results.append(extract_sql_from_response(resp))
        except Exception as e:
            print(f"      ⚠️ Greedy generation failed: {e}")
            results.append("SELECT 1")

    # ── Sampled passes (batched) ───────────────────────────────
    if n_sampled > 0:
        remaining = n_sampled
        while remaining > 0:
            bs = min(remaining, batch_size)
            batch_chats = [chat] * bs  # same prompt repeated
            try:
                inp = tokenizer(
                    batch_chats, return_tensors="pt",
                    padding=True, truncation=True,
                    max_length=MAX_SEQ_LENGTH_7B,
                ).to(model.device)
                with torch.no_grad():
                    out = model.generate(
                        **inp, max_new_tokens=300,
                        do_sample=True,
                        temperature=temperature,
                        top_p=top_p,
                        eos_token_id=tokenizer.eos_token_id,
                        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                    )
                input_len = inp.input_ids.shape[1]
                for j in range(bs):
                    resp = tokenizer.decode(
                        out[j][input_len:], skip_special_tokens=True)
                    results.append(extract_sql_from_response(resp))
            except Exception as e:
                print(f"      ⚠️ Batched generation failed (bs={bs}): {e}")
                # Fallback: fill with greedy result
                for _ in range(bs):
                    results.append(results[0] if results else "SELECT 1")
            remaining -= bs

    # Restore original padding side
    tokenizer.padding_side = orig_pad_side
    return results


def generate_single_greedy(model, tokenizer, prompt: str) -> str:
    """Single greedy generation (used for self-correction retries)."""
    return generate_candidates_batched(
        model, tokenizer, prompt,
        n_greedy=1, n_sampled=0,
    )[0]


# ═══════════════════════════════════════════════════════════════════
# TECHNIQUE 3: Self-Consistency Voting (BATCHED)
# ═══════════════════════════════════════════════════════════════════
# Generate N candidates with temperature, execute each, vote by
# execution result. Research: SC (Wang 2023), CHASE-SQL, CSC-SQL.
#
# Now uses batched generation: 1 greedy + (N-1) sampled in parallel.
# ═══════════════════════════════════════════════════════════════════

def _hash_exec_result(exec_result) -> str:
    """Hashable key from execution result for voting."""
    if exec_result is None:
        return "__NONE__"
    if isinstance(exec_result, str):
        return f"__ERR__{exec_result[:100]}"
    try:
        rows = exec_result["rows"] if isinstance(exec_result, dict) else exec_result
        def _nv(v):
            s = str(v).strip()
            try:
                f = float(s)
                return str(int(f)) if f == int(f) else f"{f:.6f}"
            except (ValueError, OverflowError):
                return s.lower()
        normalized = tuple(sorted(
            tuple(sorted(_nv(v) for v in row)) for row in rows))
        return str(normalized)
    except Exception:
        return str(exec_result)


def self_consistency_generate(model, tokenizer, sample,
                               n_candidates: int = 8,
                               temperature: float = 0.7,
                               top_p: float = 0.95,
                               use_ddl_override: bool = False,
                               ) -> Tuple[str, dict]:
    """
    Self-Consistency with execution-based voting (BATCHED).
    V4: multi-temperature + use_ddl_override.

    Multi-temperature strategy:
      Instead of sampling all candidates at one temperature, split
      across tiers (e.g., 2 at T=0.3, 3 at T=0.7, 2 at T=1.1).
      This produces both conservative near-greedy variants AND
      creative alternatives that try different SQL structures.
      Research: CHASE-SQL, Mixture-of-Temperatures.
    """
    prompt = format_tier1_prompt(sample, use_ddl_override=use_ddl_override)

    # ── Deterministic per-sample seed base (behavior-neutral addition) ──
    import zlib
    _seed_base = None
    if 'SC_DETERMINISTIC_SEEDS' in globals() and SC_DETERMINISTIC_SEEDS:
        _seed_base = SC_BASE_SEED + (zlib.crc32(str(sample.id).encode("utf-8")) % 1_000_000)

    # ── Generate candidates ──────────────────────────────────────
    # slot_info[i] = (label, temperature); logging only, does not alter voting
    slot_info = []
    if ENABLE_MULTI_TEMP:
        # Multi-temperature: 1 greedy + tiers of sampled at different T
        candidates_raw = generate_candidates_batched(
            model, tokenizer, prompt,
            n_greedy=1, n_sampled=0,
            batch_size=SC_BATCH_SIZE,
        )
        slot_info.append(("greedy", 0.0))
        temp_breakdown = {"greedy": 1}

        for tier_idx, (tier_temp, tier_count) in enumerate(SC_TEMP_TIERS):
            if tier_count <= 0:
                continue
            if _seed_base is not None:
                torch.manual_seed(_seed_base + 1000 * (tier_idx + 1))
            tier_candidates = generate_candidates_batched(
                model, tokenizer, prompt,
                n_greedy=0,
                n_sampled=tier_count,
                batch_size=SC_BATCH_SIZE,
                temperature=tier_temp,
                top_p=top_p,
            )
            candidates_raw.extend(tier_candidates)
            slot_info.extend([(f"T={tier_temp}", tier_temp)] * len(tier_candidates))
            temp_breakdown[f"T={tier_temp}"] = tier_count
    else:
        # Single temperature (original behavior)
        if _seed_base is not None:
            torch.manual_seed(_seed_base)
        candidates_raw = generate_candidates_batched(
            model, tokenizer, prompt,
            n_greedy=1,
            n_sampled=n_candidates - 1,
            batch_size=SC_BATCH_SIZE,
            temperature=temperature,
            top_p=top_p,
        )
        slot_info = [("greedy", 0.0)] + [(f"T={temperature}", temperature)] * (len(candidates_raw) - 1)
        temp_breakdown = {"greedy": 1, f"T={temperature}": n_candidates - 1}

    # ── Post-process each candidate ──
    candidates_pp = []
    for raw in candidates_raw:
        current = raw
        if ENABLE_ID_FIXING:
            current, _ = fix_sql_identifiers(current, sample.schema_ddl)
        if ENABLE_FILTER_REMOVAL:
            current, _ = remove_hallucinated_filters(current, sample.arabic_question)
        if ENABLE_SQL_VALUE_GROUNDING:
            current, _ = ground_sql_values(current, sample.db_path, sample.schema_ddl)
        candidates_pp.append(current)

    # ── Execute and vote ──
    if not sample.db_path or not os.path.exists(sample.db_path):
        return candidates_pp[0], {
            "method": "greedy_no_db",
            "n_candidates": len(candidates_raw),
        }

    result_groups = defaultdict(list)
    exec_failures = 0
    candidate_log = []   # logging only — voting below is UNCHANGED

    for _slot, sql in enumerate(candidates_pp):
        _label, _temp = slot_info[_slot] if _slot < len(slot_info) else (f"slot{_slot}", None)
        ok, result = execute_sql_on_db(sample.db_path, sql)
        _entry = {"slot": _slot, "label": _label, "temperature": _temp,
                  "raw_sql": candidates_raw[_slot] if _slot < len(candidates_raw) else None,
                  "sql": sql, "exec_ok": bool(ok)}
        if ok:
            key = _hash_exec_result(result)
            result_groups[key].append((sql, result))
            _entry["result_key"] = str(key)
            try:
                _entry["n_rows"] = len(result)
                _entry["empty_result"] = (len(result) == 0)
            except Exception:
                _entry["n_rows"] = None
                _entry["empty_result"] = None
        else:
            exec_failures += 1
            result_groups["__FAIL__"].append((sql, result))
            _entry["error"] = str(result)[:200]
        candidate_log.append(_entry)

    # Find the largest group (most common execution result)
    best_key, best_count = None, 0
    for key, group in result_groups.items():
        if key == "__FAIL__":
            continue
        if len(group) > best_count:
            best_count = len(group)
            best_key = key

    if best_key is not None:
        winner_group = result_groups[best_key]
        best_sql = min(winner_group, key=lambda x: len(x[0]))[0]
        _meta = {
            "method": "sc_vote",
            "n_candidates": len(candidates_raw),
            "n_unique_results": len(result_groups) - (1 if "__FAIL__" in result_groups else 0),
            "winner_votes": best_count,
            "exec_failures": exec_failures,
            "temp_breakdown": temp_breakdown if ENABLE_MULTI_TEMP else None,
        }
        if 'LOG_CANDIDATES' in globals() and LOG_CANDIDATES:
            _meta["candidates"] = candidate_log
        return best_sql, _meta
    else:
        # All candidates failed execution → return greedy
        _meta = {
            "method": "sc_all_failed",
            "n_candidates": len(candidates_raw),
            "exec_failures": exec_failures,
            "temp_breakdown": temp_breakdown if ENABLE_MULTI_TEMP else None,
        }
        if 'LOG_CANDIDATES' in globals() and LOG_CANDIDATES:
            _meta["candidates"] = candidate_log
        return candidates_pp[0], _meta

In [ ]:
#@title 🔑 Export Gold Result Hashes (AR) { display-mode: "form" }
import json as _json
gold_hashes = {}
_fail = 0
for _s in eval_samples:
    if _s.db_path and os.path.exists(_s.db_path):
        _ok, _r = execute_sql_on_db(_s.db_path, _s.gold_sql)
        gold_hashes[str(_s.id)] = str(_hash_exec_result(_r)) if _ok else None
        _fail += (0 if _ok else 1)
_p = "/content/gold_hashes_AR.json"
with open(_p, "w") as _f: _json.dump(gold_hashes, _f)
print(f"✅ {len(gold_hashes)} gold hashes → {_p} (gold failures: {_fail})")
try:
    import shutil as _sh; _sh.copy2(_p, "/content/drive/MyDrive/gold_hashes_AR.json"); print("✅ Drive copy")
except Exception as _e: print(f"(Drive copy skipped: {_e})")


---
## 🔄 Execution-Guided Self-Correction

If best SQL fails to execute, retry with error feedback.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# TECHNIQUE 4: Execution-Guided Self-Correction
# ═══════════════════════════════════════════════════════════════════
# If best SQL fails to execute, retry with error message.
# Research: DIN-SQL (Pourreza 2023), SQLFixAgent.
# ═══════════════════════════════════════════════════════════════════

def self_correct(model, tokenizer, sample, failed_sql: str,
                 error_msg: str, max_retries: int = 2) -> Tuple[str, int]:
    """Retry SQL generation with error feedback."""
    if ENABLE_MSCHEMA:
        schema_str = get_mschema(sample.db_id, sample.schema_ddl)
    else:
        schema_str = sample.schema_ddl[:4500]

    value_hints = ""
    if ENABLE_VALUE_HINTS and sample.db_path:
        value_hints = sample_db_values_question_aware(
            sample.db_path, sample.schema_ddl,
            sample.arabic_question, max_per_col=VALUE_HINT_MAX_PER_COL)

    current_sql = failed_sql
    current_error = error_msg

    for attempt in range(max_retries):
        retry_prompt = TIER1_RETRY_PROMPT.format(
            schema=schema_str, value_hints=value_hints,
            failed_sql=current_sql[:200], error_msg=current_error[:150],
            question=sample.arabic_question)
        corrected = generate_single_greedy(model, tokenizer, retry_prompt)
        if ENABLE_ID_FIXING:
            corrected, _ = fix_sql_identifiers(corrected, sample.schema_ddl)
        if ENABLE_SQL_VALUE_GROUNDING:
            corrected, _ = ground_sql_values(corrected, sample.db_path, sample.schema_ddl)
        if sample.db_path and os.path.exists(sample.db_path):
            ok, result = execute_sql_on_db(sample.db_path, corrected)
            if ok:
                return corrected, attempt + 1
            else:
                current_sql = corrected
                current_error = str(result)
        else:
            return corrected, attempt + 1
    return current_sql, max_retries

---
## 🔄 Combined Pipeline + Evaluation Function

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# COMBINED PIPELINE
# ═══════════════════════════════════════════════════════════════════

def evaluate_single_sample(model, tokenizer, sample, idx: int) -> dict:
    """Full Tier 1 pipeline for one sample — V4 with low-confidence fallback."""
    metadata = {}

    # Step 1: Generate (batched SC or single greedy)
    if ENABLE_SELF_CONSISTENCY:
        pred_sql, sc_meta = self_consistency_generate(
            model, tokenizer, sample,
            n_candidates=SC_NUM_CANDIDATES,
            temperature=SC_TEMPERATURE, top_p=SC_TOP_P)
        metadata.update(sc_meta)

        # V4: Low-confidence fallback — re-run with DDL + extra candidates
        if (ENABLE_LOW_CONF_FALLBACK
            and sc_meta.get("winner_votes", 99) <= LOW_CONF_VOTE_THRESHOLD):

            pred_sql_2, sc_meta_2 = self_consistency_generate(
                model, tokenizer, sample,
                n_candidates=SC_NUM_CANDIDATES + LOW_CONF_EXTRA_CANDIDATES,
                temperature=SC_TEMPERATURE * 0.85,
                top_p=SC_TOP_P,
                use_ddl_override=True,
            )
            votes_1 = sc_meta.get("winner_votes", 0)
            votes_2 = sc_meta_2.get("winner_votes", 0)
            total_1 = sc_meta.get("n_candidates", 1)
            total_2 = sc_meta_2.get("n_candidates", 1)
            conf_1 = votes_1 / total_1
            conf_2 = votes_2 / total_2

            if conf_2 > conf_1:
                pred_sql = pred_sql_2
                metadata.update(sc_meta_2)
                metadata["fallback_used"] = True
                metadata["fallback_reason"] = f"conf {conf_1:.2f}->{conf_2:.2f}"
            else:
                metadata["fallback_used"] = False
                metadata["fallback_reason"] = f"kept orig {conf_1:.2f}>={conf_2:.2f}"
    else:
        prompt = format_tier1_prompt(sample)
        raw_sql = generate_single_greedy(model, tokenizer, prompt)
        pred_sql = raw_sql
        if ENABLE_ID_FIXING:
            pred_sql, idf = fix_sql_identifiers(pred_sql, sample.schema_ddl)
            metadata["id_fixes"] = len(idf)
        if ENABLE_FILTER_REMOVAL:
            pred_sql, fr = remove_hallucinated_filters(pred_sql, sample.arabic_question)
            metadata["filter_removals"] = len(fr)
        if ENABLE_SQL_VALUE_GROUNDING:
            pred_sql, vg = ground_sql_values(pred_sql, sample.db_path, sample.schema_ddl)
            metadata["value_groundings"] = len(vg)
        metadata["method"] = "single"

    # Step 2: Self-correction (if execution fails)
    if ENABLE_SELF_CORRECTION and sample.db_path and os.path.exists(sample.db_path):
        ok, result = execute_sql_on_db(sample.db_path, pred_sql)
        if not ok:
            corrected, retries = self_correct(
                model, tokenizer, sample,
                pred_sql, str(result), max_retries=SC_MAX_RETRIES)
            if corrected != pred_sql:
                pred_sql = corrected
                metadata["self_corrected"] = True
                metadata["retries_used"] = retries

    # Step 3: Compute metrics
    ex = None
    if sample.db_path and os.path.exists(sample.db_path):
        gold_ok, gold_res = execute_sql_on_db(sample.db_path, sample.gold_sql)
        pred_ok, pred_res = execute_sql_on_db(sample.db_path, pred_sql)
        if gold_ok and pred_ok:
            ex = compare_results(gold_res, pred_res)
        elif gold_ok and not pred_ok:
            ex = False

    em = compute_exact_match(pred_sql, sample.gold_sql)
    bleu = compute_bleu(pred_sql, sample.gold_sql)
    sqam = compute_sqam(pred_sql, sample.gold_sql)
    tsed = compute_tsed(pred_sql, sample.gold_sql)

    return {
        "id": sample.id, "db_id": sample.db_id,
        "question": sample.arabic_question,
        "gold_sql": sample.gold_sql, "pred_sql": pred_sql,
        "ex": ex, "em": em,
        "bleu": bleu, "sqam": sqam, "tsed": tsed,
        "metadata": metadata,
    }

---
## 🧪 Experiment Suite

The cells below define, run, and summarise the ablation suite using the functions defined above.

* **Voting** = 8-candidate multi-temperature self-consistency.
* **Repair stage** = identifier fixing, filter removal, value grounding, self-correction, low-confidence fallback.
* **Schema grounding** = schema representation, column-linking hints, native column descriptions / table glosses.
* **Value grounding** = value hints, matched values, sample rows, repair-stage value grounding.
* **Translation** = external English translation (prompt, entity gating, value matching).

Metrics per configuration: EX, official Spider ESM, EM, per-hardness breakdown, 95% bootstrap CIs, McNemar tests against `full_system`.


In [ ]:
#@title 🧪 Experiment Suite — Configuration { display-mode: "form" }
import json, os, time, math, sys, re, copy, subprocess, importlib, platform, shutil, hashlib, traceback
from collections import OrderedDict, defaultdict
from datetime import datetime, timezone

LANGUAGE     = "Arabic"
LANG_CODE    = "ar"
BENCHMARK    = "Ar-Spider dev"
Q_ATTR       = "arabic_question"            # question attribute on EvalSample
DESC_FLAG    = "ENABLE_AR_COL_DESC"         # native column-description toggle (column hints / embeddings)
DATASET_DIR  = AR_SPIDER_DIR     # folder holding tables.json + database/

EXP_ROOT      = "/content/drive/MyDrive/t2s_ablation_suite"   #@param {type:"string"}
SMOKE_TEST_N  = 0        #@param {type:"integer"}
RUN_CONFIGS   = "all"    #@param {type:"string"}
INCLUDE_CANDIDATE_LOGS_IN_MASTER = False   #@param {type:"boolean"}
PROGRESS_EVERY = 25      #@param {type:"integer"}

SYNC_EVERY    = 20       #@param {type:"integer"}

_sub = LANG_CODE if SMOKE_TEST_N == 0 else f"{LANG_CODE}_smoke{SMOKE_TEST_N}"
EXP_DIR   = os.path.join(EXP_ROOT, _sub)                       # Drive copy (durable)
LOCAL_DIR = os.path.join("/content/t2s_ablation_local", _sub)   # local working copy (fast, survives Drive hiccups)
os.makedirs(LOCAL_DIR, exist_ok=True)
try:
    os.makedirs(EXP_DIR, exist_ok=True)
except OSError as _e:
    print(f"⚠️ Drive folder not reachable right now ({_e}); results will be kept locally and synced when Drive is back")
MASTER_JSON = os.path.join(EXP_DIR, f"ablation_suite_{LANG_CODE}.json")
MASTER_JSON_LOCAL = os.path.join(LOCAL_DIR, f"ablation_suite_{LANG_CODE}.json")

# ── Toggles that define a configuration (all must be True in the loaded notebook = full system) ──
FLAG_NAMES = [
    "ENABLE_MSCHEMA", "ENABLE_VALUE_HINTS", "ENABLE_COLUMN_LINKING", DESC_FLAG,
    "ENABLE_ENGLISH_HINT", "ENABLE_BILINGUAL_SCHEMA", "ENABLE_VALUE_INJECTION",
    "ENABLE_BILINGUAL_EMBEDDINGS", "ENABLE_SAMPLE_ROWS",
    "ENABLE_SELF_CONSISTENCY", "ENABLE_MULTI_TEMP", "ENABLE_LOW_CONF_FALLBACK",
    "ENABLE_SELF_CORRECTION", "ENABLE_ID_FIXING", "ENABLE_FILTER_REMOVAL", "ENABLE_SQL_VALUE_GROUNDING",
]
DEFAULT_FLAGS = {k: bool(globals()[k]) for k in FLAG_NAMES}
DEFAULT_TOP_K = int(COL_LINK_TOP_K)
_off = [k for k, v in DEFAULT_FLAGS.items() if not v]
assert not _off, f"The notebook must be loaded in full-system mode before running the suite; currently OFF: {_off}"

REPAIR_OFF = {"ENABLE_ID_FIXING": False, "ENABLE_FILTER_REMOVAL": False, "ENABLE_SQL_VALUE_GROUNDING": False,
              "ENABLE_SELF_CORRECTION": False, "ENABLE_LOW_CONF_FALLBACK": False}
VOTING_OFF = {"ENABLE_SELF_CONSISTENCY": False}   # single greedy decode; the fallback is unreachable without voting
SCHEMA_GROUNDING_OFF = {"ENABLE_MSCHEMA": False, "ENABLE_COLUMN_LINKING": False, DESC_FLAG: False,
                        "ENABLE_BILINGUAL_SCHEMA": False, "ENABLE_BILINGUAL_EMBEDDINGS": False}
VALUE_GROUNDING_OFF  = {"ENABLE_VALUE_HINTS": False, "ENABLE_VALUE_INJECTION": False, "ENABLE_SAMPLE_ROWS": False,
                        "ENABLE_SQL_VALUE_GROUNDING": False}
NATIVE_DESC_OFF      = {DESC_FLAG: False, "ENABLE_BILINGUAL_SCHEMA": False, "ENABLE_BILINGUAL_EMBEDDINGS": False}

# Special key "translation": "full" | "none"
CONFIGS = OrderedDict([
    ("greedy_control",           dict(row="1", ref="grouped G4",
                                     desc="Voting OFF, repair OFF (single greedy decode)",
                                     overrides={**VOTING_OFF, **REPAIR_OFF})),
    ("baseline_minimal",         dict(row="5", ref="minimal baseline",
                                     desc="Raw DDL + question; greedy; no repair",
                                     overrides={**SCHEMA_GROUNDING_OFF, **VALUE_GROUNDING_OFF, **VOTING_OFF, **REPAIR_OFF})),
    ("abl_no_voting",            dict(row="3", ref="single",
                                     desc="− Self-consistency voting",
                                     overrides={**VOTING_OFF})),
    ("full_system",              dict(row="2", ref="full system",
                                     desc="Complete pipeline",
                                     overrides={})),
    ("abl_no_repair",            dict(row="3", ref="single",
                                     desc="− Repair stage",
                                     overrides={**REPAIR_OFF})),
    ("abl_no_column_linking",    dict(row="3", ref="single",
                                     desc="− Column-linking hints",
                                     overrides={"ENABLE_COLUMN_LINKING": False})),
    ("abl_no_value_injection",   dict(row="3", ref="single",
                                     desc="− Value injection",
                                     overrides={"ENABLE_VALUE_HINTS": False, "ENABLE_VALUE_INJECTION": False})),
    ("abl_no_sample_rows",       dict(row="3", ref="single",
                                     desc="− Sample rows",
                                     overrides={"ENABLE_SAMPLE_ROWS": False})),
    ("grp_no_schema_grounding",  dict(row="4", ref="grouped G1",
                                     desc="G1: all schema-grounding components removed",
                                     overrides={**SCHEMA_GROUNDING_OFF})),
    ("grp_no_value_grounding",   dict(row="4", ref="grouped G2",
                                     desc="G2: all value-grounding components removed",
                                     overrides={**VALUE_GROUNDING_OFF})),
    ("grp_no_translation",       dict(row="4", ref="grouped G3",
                                     desc="G3: external translation removed",
                                     overrides={"translation": "none", **NATIVE_DESC_OFF})),
])

SELECTED = list(CONFIGS) if RUN_CONFIGS.strip().lower() == "all" else [c.strip() for c in RUN_CONFIGS.split(",") if c.strip()]
_unknown = [c for c in SELECTED if c not in CONFIGS]
assert not _unknown, f"Unknown config name(s): {_unknown}. Valid: {list(CONFIGS)}"

def _is_greedy(cfg):
    return cfg["overrides"].get("ENABLE_SELF_CONSISTENCY", True) is False

print(f"🧪 {LANGUAGE} ablation suite → {EXP_DIR}")
print(f"   local working copy: {LOCAL_DIR}  (synced to Drive every {SYNC_EVERY} samples)")
print(f"   master JSON: {MASTER_JSON}")
print(f"   {'SMOKE TEST: first ' + str(SMOKE_TEST_N) + ' samples' if SMOKE_TEST_N else 'full dev set'}")
print(f"\n   {'#':<3} {'config':<26} {'row':<4} {'decode':<7} description")
for i, name in enumerate(SELECTED, 1):
    cfg = CONFIGS[name]
    print(f"   {i:<3} {name:<26} {cfg['row']:<4} {'greedy' if _is_greedy(cfg) else '8-cand':<7} {cfg['desc']}")
n_full = sum(1 for n in SELECTED if not _is_greedy(CONFIGS[n])); n_greedy = len(SELECTED) - n_full
print(f"\n   Rough cost at ~29 s/sample (L4) for 8-candidate runs and ~4 s/sample for greedy runs:"
      f" {n_full} × 8.3 h + {n_greedy} × 1.2 h ≈ {n_full*8.3 + n_greedy*1.2:.0f} h on an L4 (A100: ~3× faster). Resumable.")


In [ ]:
#@title 🧪 Experiment Suite — Helpers (official ESM, CIs, McNemar, resumable runner) { display-mode: "form" }
import numpy as np
import torch

# ── 0. Dev set hygiene: drop padding rows with empty gold (the MultiSpider loader pads to 1,200) ──
_kept = [s for s in eval_samples if s.gold_sql and s.gold_sql.strip()]
if len(_kept) != len(eval_samples):
    print(f"ℹ️ Dropped {len(eval_samples) - len(_kept)} rows with empty gold SQL")
eval_samples = _kept
SUITE_SAMPLES = eval_samples[:SMOKE_TEST_N] if SMOKE_TEST_N > 0 else eval_samples
SUITE_IDS = [s.id for s in SUITE_SAMPLES]
print(f"✅ Suite evaluates {len(SUITE_SAMPLES)} samples of {BENCHMARK}")
if SMOKE_TEST_N == 0 and len(SUITE_SAMPLES) != 1034:
    print(f"⚠️ Expected 1,034 dev samples, found {len(SUITE_SAMPLES)} — check the dataset loader before running")

# ── 1. Official Spider ESM, per sample (taoyds/spider evaluation.py, --etype match) ──
SPIDER_EVAL_DIR = "/content/spider_eval"
TABLES_JSON = os.path.join(DATASET_DIR, "tables.json")
DB_DIR = next((os.path.join(DATASET_DIR, c) for c in ["database", "databases"]
               if os.path.isdir(os.path.join(DATASET_DIR, c))), "")
assert os.path.exists(TABLES_JSON), f"tables.json not found at {TABLES_JSON}"
assert DB_DIR, f"database directory not found under {DATASET_DIR}"

_EMPTY_SQL = {"except": None, "from": {"conds": [], "table_units": []}, "groupBy": [], "having": [],
              "intersect": None, "limit": None, "orderBy": [], "select": [False, []], "union": None, "where": []}
_spider_eval = None
_schema_cache = {}

def _setup_spider_eval():
    global _spider_eval
    if _spider_eval is not None:
        return _spider_eval
    os.makedirs(SPIDER_EVAL_DIR, exist_ok=True)
    repo = os.path.join(SPIDER_EVAL_DIR, "spider_repo")
    if not os.path.exists(os.path.join(SPIDER_EVAL_DIR, "evaluation.py")):
        print("📥 Cloning taoyds/spider for the official evaluator ...")
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/taoyds/spider.git", repo],
                       check=True, capture_output=True)
        for fn in ["evaluation.py", "process_sql.py"]:
            shutil.copy2(os.path.join(repo, fn), os.path.join(SPIDER_EVAL_DIR, fn))
    commit = None
    try:
        commit = subprocess.run(["git", "-C", repo, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    except Exception:
        pass
    if SPIDER_EVAL_DIR not in sys.path:
        sys.path.insert(0, SPIDER_EVAL_DIR)
    ev = importlib.import_module("evaluation")
    ps = importlib.import_module("process_sql")
    kmaps = ev.build_foreign_key_map_from_json(TABLES_JSON)
    _spider_eval = (ev, ps, kmaps, commit)
    print(f"✅ Official Spider evaluator ready (commit {commit})")
    return _spider_eval

def _one_line(sql):
    return " ".join(sql.strip().rstrip(";").split())

def official_esm_per_sample(records):
    """Per-sample exact-set-match verdicts and hardness with the official evaluator (values/DISTINCT disabled)."""
    ev, ps, kmaps, _ = _setup_spider_eval()
    evaluator = ev.Evaluator()
    verdicts, hardness = [], []
    for r in records:
        db_id = r["db_id"]
        try:
            if db_id not in _schema_cache:
                _schema_cache[db_id] = ps.Schema(ps.get_schema(os.path.join(DB_DIR, db_id, db_id + ".sqlite")))
            schema = _schema_cache[db_id]
            g_sql = ps.get_sql(schema, _one_line(r["gold_sql"]))
            h = evaluator.eval_hardness(g_sql)
            try:
                p_sql = ps.get_sql(schema, _one_line(r["pred_sql"]))
            except Exception:
                p_sql = copy.deepcopy(_EMPTY_SQL)
            kmap = kmaps[db_id]
            g_valid = ev.build_valid_col_units(g_sql["from"]["table_units"], schema)
            g_sql = ev.rebuild_sql_col(g_valid, ev.rebuild_sql_val(g_sql), kmap)
            p_valid = ev.build_valid_col_units(p_sql["from"]["table_units"], schema)
            p_sql = ev.rebuild_sql_col(p_valid, ev.rebuild_sql_val(p_sql), kmap)
            verdicts.append(bool(evaluator.eval_exact_match(p_sql, g_sql)))
            hardness.append(h)
        except Exception:
            verdicts.append(None)
            hardness.append(None)
    return verdicts, hardness

def official_esm_script(records):
    """Cross-check: aggregate ESM from running evaluation.py as a script (same as the original notebook cell)."""
    try:
        _setup_spider_eval()
        gold_p = os.path.join(SPIDER_EVAL_DIR, "gold_tmp.sql"); pred_p = os.path.join(SPIDER_EVAL_DIR, "pred_tmp.sql")
        with open(gold_p, "w", encoding="utf-8") as gf, open(pred_p, "w", encoding="utf-8") as pf:
            for r in records:
                gf.write(f"{_one_line(r['gold_sql'])}\t{r['db_id']}\n"); pf.write(f"{_one_line(r['pred_sql'])}\n")
        out = subprocess.run([sys.executable, os.path.join(SPIDER_EVAL_DIR, "evaluation.py"), "--gold", gold_p,
                              "--pred", pred_p, "--etype", "match", "--db", DB_DIR, "--table", TABLES_JSON],
                             capture_output=True, text=True, cwd=SPIDER_EVAL_DIR)
        for line in (out.stdout + "\n" + out.stderr).splitlines():
            if line.strip().lower().startswith("exact match"):
                nums = re.findall(r"\d+\.\d+", line)
                if nums:
                    return round(100 * float(nums[-1]), 2)   # last column = "all"
    except Exception as e:
        print(f"   (script cross-check skipped: {e})")
    return None

# ── 2. Statistics ──
def pct(x, n): return round(100.0 * x / n, 2) if n else None

def bootstrap_ci(outcomes, B=10000, seed=0):
    """95% percentile bootstrap CI of a proportion. Resampling n Bernoulli(p̂) outcomes with replacement
    is exactly Binomial(n, p̂), so the bootstrap distribution is drawn directly."""
    x = np.array([1 if o else 0 for o in outcomes]); n = len(x)
    if n == 0: return None
    p = x.mean(); rng = np.random.default_rng(seed)
    means = rng.binomial(n, p, size=B) / n
    return [round(100 * float(np.percentile(means, 2.5)), 2), round(100 * float(np.percentile(means, 97.5)), 2)]

def mcnemar_exact(ref, cmp):
    """Two-sided exact McNemar test on paired binary outcomes (None treated as incorrect)."""
    b = sum(1 for x, y in zip(ref, cmp) if bool(x) and not bool(y))   # ref correct only
    c = sum(1 for x, y in zip(ref, cmp) if not bool(x) and bool(y))   # cmp correct only
    n = b + c
    if n == 0:
        return {"ref_only": 0, "cmp_only": 0, "n_discordant": 0, "p": 1.0}
    k = min(b, c)
    try:
        from scipy.stats import binomtest
        p = float(binomtest(k, n, 0.5, alternative="two-sided").pvalue)
    except Exception:
        from math import lgamma, exp, log
        lp = lambda i: lgamma(n + 1) - lgamma(i + 1) - lgamma(n - i + 1) + n * log(0.5)
        p = min(1.0, 2 * sum(exp(lp(i)) for i in range(0, k + 1)))
    return {"ref_only": b, "cmp_only": c, "n_discordant": n, "p": p}

# ── 3. Configuration switching ──
_ORIG_ENGLISH = {s.id: s.english_question for s in eval_samples}
if "_orig_format_tier1_prompt" not in globals():
    _orig_format_tier1_prompt = format_tier1_prompt          # captured once (safe to re-run this cell)
_HINT_LINE_RE = re.compile(r"^\(English: .*\)\n", re.M)

def format_tier1_prompt(sample, use_ddl_override=False, _orig=_orig_format_tier1_prompt):
    """Suite wrapper: identical to the original builder, except that the '(English: …)' line is
    removed when ENABLE_ENGLISH_HINT is False."""
    p = _orig(sample, use_ddl_override=use_ddl_override)
    if not ENABLE_ENGLISH_HINT:
        p = _HINT_LINE_RE.sub("", p, count=1)
    return p

_col_cache_state = {"bilingual": DEFAULT_FLAGS["ENABLE_BILINGUAL_EMBEDDINGS"]}
def _ensure_col_cache():
    """Column-linking embeddings include native descriptions when ENABLE_BILINGUAL_EMBEDDINGS; rebuild on change."""
    want = bool(globals()["ENABLE_BILINGUAL_EMBEDDINGS"])
    if globals()["ENABLE_COLUMN_LINKING"] and _col_cache_state["bilingual"] != want:
        print(f"   🔄 Rebuilding column embeddings (bilingual={want}) ...")
        _col_emb_cache.clear(); prewarm_column_cache(); _col_cache_state["bilingual"] = want

def apply_config(overrides):
    g = globals()
    for k, v in DEFAULT_FLAGS.items():
        g[k] = v
    for s in eval_samples:
        s.english_question = _ORIG_ENGLISH[s.id]
    for k, v in overrides.items():
        if k == "translation":
            if v == "none":
                g["ENABLE_ENGLISH_HINT"] = False
                for s in eval_samples:
                    s.english_question = ""
            elif v != "full":
                raise ValueError(f"unknown translation mode {v}")
        else:
            assert k in DEFAULT_FLAGS, f"unknown flag {k}"
            g[k] = bool(v)
    _ensure_col_cache()
    return {k: bool(g[k]) for k in FLAG_NAMES} | {"translation": overrides.get("translation", "full")}

# ── 4. Record slimming (keep candidate logs, but replace result dumps by short hashes) ──
def _slim_record(rec):
    meta = rec.get("metadata", {})
    for c in meta.get("candidates", []) or []:
        rk = c.get("result_key")
        if isinstance(rk, str):
            c["result_key"] = hashlib.sha1(rk.encode("utf-8")).hexdigest()[:16]
        for f in ("raw_sql", "sql", "error"):
            if isinstance(c.get(f), str) and len(c[f]) > 600:
                c[f] = c[f][:600] + "…"
    return rec

def _load_jsonl(path):
    recs = {}
    try:
        if os.path.exists(path):
            with open(path, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    try:
                        r = json.loads(line); recs[r["id"]] = r
                    except Exception:
                        pass
    except OSError:
        pass
    return recs

# ── 4b. Drive-safe storage: work locally, mirror to Drive ──
_SUFFIXES = (".jsonl", ".config.json", ".esm.json")

def _drive_ok():
    try:
        os.listdir(EXP_DIR); return True
    except OSError:
        return False

def _remount_drive():
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive", force_remount=True)
        os.makedirs(EXP_DIR, exist_ok=True)
    except Exception as _e:
        print(f"   ⚠️ Drive remount failed: {_e}")
    return _drive_ok()

def _write_records(path, recs_by_id):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for i in SUITE_IDS:
            if i in recs_by_id:
                f.write(json.dumps(recs_by_id[i], ensure_ascii=False, default=str) + "\n")
        for i, r in recs_by_id.items():           # anything not in the suite order (should not happen)
            if i not in SUITE_IDS:
                f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
    os.replace(tmp, path)

def _pull_from_drive(verbose=True):
    """Merge whatever exists on Drive into the local working copy (union of sample ids)."""
    if not _drive_ok():
        if verbose: print("   ℹ️ Drive not reachable; using local copy only")
        return
    pulled = 0
    for fn in os.listdir(EXP_DIR):
        if not fn.endswith(_SUFFIXES): continue
        src, dst = os.path.join(EXP_DIR, fn), os.path.join(LOCAL_DIR, fn)
        try:
            if fn.endswith(".jsonl"):
                merged = _load_jsonl(dst); before = len(merged)
                merged.update({k: v for k, v in _load_jsonl(src).items() if k not in merged})
                if len(merged) != before or not os.path.exists(dst):
                    _write_records(dst, merged); pulled += 1
            elif not os.path.exists(dst):
                shutil.copy2(src, dst); pulled += 1
        except OSError as _e:
            print(f"   ⚠️ could not pull {fn}: {_e}")
    if verbose: print(f"   ⤵ pulled {pulled} file(s) from Drive into {LOCAL_DIR}")

def _push_to_drive(name=None, quiet=False):
    """Atomically copy local files (one config, or all) to Drive. Returns True on success."""
    if not _drive_ok() and not _remount_drive():
        if not quiet: print("   ⚠️ Drive unavailable — progress kept locally; will retry at the next sync")
        return False
    ok = True
    for fn in os.listdir(LOCAL_DIR):
        if not fn.endswith(_SUFFIXES + (".json",)): continue
        if name and not fn.startswith(name + "."): continue
        src, dst = os.path.join(LOCAL_DIR, fn), os.path.join(EXP_DIR, fn)
        try:
            shutil.copy2(src, dst + ".tmp"); os.replace(dst + ".tmp", dst)
        except OSError as _e:
            ok = False
            if not quiet: print(f"   ⚠️ sync of {fn} failed: {_e}")
    return ok

_pull_from_drive()

# ── 5. Resumable runner ──
def run_config(name):
    cfg = CONFIGS[name]
    _pull_from_drive(verbose=False)
    jsonl = os.path.join(LOCAL_DIR, f"{name}.jsonl")
    done = _load_jsonl(jsonl)
    todo = [s for s in SUITE_SAMPLES if s.id not in done]
    print("=" * 78)
    print(f"▶ {name}  [{cfg['ref']}]  {cfg['desc']}")
    print(f"  {len(done)} done, {len(todo)} to run → {jsonl}  (mirrored to {EXP_DIR})")
    if not todo:
        print("  ✅ already complete"); return
    resolved = apply_config(cfg["overrides"])
    print("  flags OFF: " + (", ".join(k for k, v in resolved.items() if v is False) or "none")
          + f" | translation={resolved['translation']}")
    with open(os.path.join(LOCAL_DIR, f"{name}.config.json"), "w") as f:
        json.dump({"name": name, **cfg, "resolved_flags": resolved}, f, indent=2, ensure_ascii=False)
    _push_to_drive(name, quiet=True)
    t0 = time.time(); n_ok = 0
    try:
        with open(jsonl, "a", encoding="utf-8") as f:
            for i, s in enumerate(todo):
                rec = evaluate_single_sample(infer_model, infer_tok, s, i)
                rec = _slim_record(rec)
                rec["config"] = name; rec["ts"] = round(time.time(), 1)
                f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n"); f.flush()
                n_ok += 1 if rec["ex"] is True else 0
                if i < 3 or (i + 1) % PROGRESS_EVERY == 0 or i + 1 == len(todo):
                    el = time.time() - t0; rate = el / (i + 1); eta = rate * (len(todo) - i - 1)
                    print(f"  [{len(done)+i+1:>4}/{len(SUITE_SAMPLES)}] EX(run)={100*n_ok/(i+1):5.1f}%"
                          f"  {rate:4.1f}s/sample  ETA {eta/60:5.1f} min  | {rec['pred_sql'][:70]}")
                if (i + 1) % SYNC_EVERY == 0:
                    _push_to_drive(name, quiet=False)
                if (i + 1) % 100 == 0 and torch.cuda.is_available():
                    torch.cuda.empty_cache()
    finally:
        apply_config({})   # always restore the full-system defaults
        _push_to_drive(name, quiet=False)
    print(f"  ✅ {name} finished: {len(todo)} samples in {(time.time()-t0)/60:.1f} min")

# ── 6. Master JSON ──
def _versions():
    v = {"python": platform.python_version(), "sqlite": __import__("sqlite3").sqlite_version}
    for m in ["torch", "transformers", "peft", "bitsandbytes", "sentence_transformers", "accelerate"]:
        try: v[m] = importlib.import_module(m).__version__
        except Exception: v[m] = None
    return v

def _provenance():
    try:
        _commit = _setup_spider_eval()[3]
    except Exception:
        _commit = None
    return {
        "language": LANGUAGE, "benchmark": BENCHMARK, "base_model": BASE_MODEL, "adapter_path": DRIVE_SAVE_DIR,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "versions": _versions(), "spider_eval_commit": _commit,
        "sampling_base_seed": SC_BASE_SEED, "deterministic_seeds": bool(SC_DETERMINISTIC_SEEDS),
        "temperature_schedule": [["greedy", 1]] + [[t, c] for t, c in SC_TEMP_TIERS],
        "n_candidates": 1 + sum(c for _, c in SC_TEMP_TIERS), "top_p": SC_TOP_P, "max_new_tokens": 300,
        "constants": {"col_link_threshold": 0.50, "col_link_top_k": COL_LINK_TOP_K,
                      "value_hint_max_per_col": VALUE_HINT_MAX_PER_COL,
                      "matched_values_top_k": VALUE_INJECTION_TOP_K, "matched_values_threshold": VALUE_INJECTION_THRESHOLD,
                      "sample_rows_limit": SAMPLE_ROWS_LIMIT, "sample_rows_max_tables": SAMPLE_ROWS_MAX_TABLES,
                      "sql_value_grounding_threshold": SQL_VALUE_SIM_THRESHOLD,
                      "low_conf_vote_threshold": LOW_CONF_VOTE_THRESHOLD, "self_correction_max_retries": SC_MAX_RETRIES},
        "ex_definition": "notebook comparator compare_results (row-order-insensitive; within-row value sorting; case-insensitive strings; numeric normalisation); EX% uses all dev samples as denominator",
        "esm_definition": "taoyds/spider evaluation.py --etype match (DISABLE_VALUE, DISABLE_DISTINCT), computed per sample",
        "ablation_design": "inference-time removal of components from the fixed per-language adapter (trained with per-feature prompt dropout); no retraining",
        "suite_samples": len(SUITE_SAMPLES), "smoke_test_n": SMOKE_TEST_N,
    }

def _summarise(name, recs_by_id):
    recs = [recs_by_id[i] for i in SUITE_IDS if i in recs_by_id]
    n = len(recs)
    _sig = hashlib.sha1("\n".join(r["id"] + "\t" + r["pred_sql"] for r in recs).encode("utf-8")).hexdigest()
    _cache_p = os.path.join(LOCAL_DIR, f"{name}.esm.json")
    esm = hard = None
    if os.path.exists(_cache_p):
        try:
            _c = json.load(open(_cache_p))
            if _c.get("sig") == _sig:
                esm, hard = _c["esm"], _c["hardness"]
        except Exception:
            pass
    if esm is None:
        try:
            esm, hard = official_esm_per_sample(recs)
            json.dump({"sig": _sig, "esm": esm, "hardness": hard}, open(_cache_p, "w"))
        except Exception as _e:
            print(f"   ⚠️ official ESM unavailable for {name}: {_e}")
            esm, hard = [None] * n, [None] * n
    ex = [r["ex"] for r in recs]
    em = [bool(r.get("em")) for r in recs]
    metas = [r.get("metadata", {}) or {} for r in recs]
    by_h = defaultdict(lambda: {"n": 0, "ex": 0, "esm": 0})
    for r, e, h in zip(recs, esm, hard):
        k = h or "unknown"; by_h[k]["n"] += 1; by_h[k]["ex"] += 1 if r["ex"] is True else 0; by_h[k]["esm"] += 1 if e else 0
    votes = defaultdict(int)
    for m in metas:
        if m.get("winner_votes") is not None: votes[str(m["winner_votes"])] += 1
    cand_total = sum(m.get("n_candidates", 0) for m in metas)
    cand_fail = sum(m.get("exec_failures", 0) for m in metas)
    ts = [r.get("ts") for r in recs if r.get("ts")]
    summary = {
        "status": "complete" if n == len(SUITE_SAMPLES) else f"partial ({n}/{len(SUITE_SAMPLES)})",
        "n": n,
        "EX": pct(sum(1 for e in ex if e is True), n),
        "EX_count": f"{sum(1 for e in ex if e is True)}/{n}",
        "EX_over_evaluable": pct(sum(1 for e in ex if e is True), sum(1 for e in ex if e is not None)),
        "EX_ci95": bootstrap_ci([e is True for e in ex]),
        "ESM_official": pct(sum(1 for e in esm if e), n),
        "ESM_official_count": f"{sum(1 for e in esm if e)}/{n}",
        "ESM_ci95": bootstrap_ci([bool(e) for e in esm]),
        "ESM_unparsed": sum(1 for e in esm if e is None),
        "EM_string": pct(sum(em), n),
        "by_hardness": {k: {"n": v["n"], "EX": pct(v["ex"], v["n"]), "ESM": pct(v["esm"], v["n"])} for k, v in sorted(by_h.items())},
        "stats": {
            "winner_votes_hist": dict(sorted(votes.items(), key=lambda kv: int(kv[0]))),
            "all_candidates_failed": sum(1 for m in metas if m.get("method") == "sc_all_failed"),
            "candidate_exec_failure_rate": pct(cand_fail, cand_total) if cand_total else None,
            "fallback_attempted": sum(1 for m in metas if "fallback_used" in m),
            "fallback_retained": sum(1 for m in metas if m.get("fallback_used") is True),
            "self_corrected": sum(1 for m in metas if m.get("self_corrected")),
            "gold_execution_failures": sum(1 for e in ex if e is None),
            "wall_clock_h": round((max(ts) - min(ts)) / 3600, 2) if len(ts) > 1 else None,
        },
    }
    mc = defaultdict(int)
    for m in metas: mc[m.get("method", "?")] += 1
    summary["stats"]["method_counts"] = dict(mc)
    samples = []
    for r, e, h in zip(recs, esm, hard):
        m = r.get("metadata", {}) or {}
        row = {"id": r["id"], "db_id": r["db_id"], "hardness": h, "gold_sql": r["gold_sql"], "pred_sql": r["pred_sql"],
               "ex": r["ex"], "esm": e, "em": bool(r.get("em")), "method": m.get("method"),
               "winner_votes": m.get("winner_votes"), "n_candidates": m.get("n_candidates"),
               "exec_failures": m.get("exec_failures"), "fallback_used": m.get("fallback_used"),
               "self_corrected": bool(m.get("self_corrected", False))}
        if INCLUDE_CANDIDATE_LOGS_IN_MASTER and m.get("candidates"):
            row["candidates"] = m["candidates"]
        samples.append(row)
    return summary, samples, {"ex": [e is True for e in ex], "esm": [bool(e) for e in esm]}

def build_master(write=True, verbose=True):
    _pull_from_drive(verbose=False)
    master = {"suite": "inference-time ablation suite", "created": datetime.now(timezone.utc).isoformat(),
              "provenance": _provenance(), "configs": OrderedDict(), "comparisons_vs_full_system": OrderedDict(),
              "redundancy_analysis": {}, "notes": []}
    outcomes = {}
    for name, cfg in CONFIGS.items():
        recs = _load_jsonl(os.path.join(LOCAL_DIR, f"{name}.jsonl"))
        if not recs: continue
        summary, samples, outc = _summarise(name, recs)
        if summary["status"] == "complete":
            summary["ESM_official_script_crosscheck"] = official_esm_script([recs[i] for i in SUITE_IDS])
        cfgpath = os.path.join(LOCAL_DIR, f"{name}.config.json")
        resolved = json.load(open(cfgpath)).get("resolved_flags") if os.path.exists(cfgpath) else None
        master["configs"][name] = {"row": cfg["row"], "ref": cfg["ref"], "description": cfg["desc"],
                                   "overrides": {k: v for k, v in cfg["overrides"].items()}, "resolved_flags": resolved,
                                   "metrics": summary, "samples": samples}
        outcomes[name] = outc
    full = outcomes.get("full_system")
    if full and master["configs"]["full_system"]["metrics"]["status"] == "complete":
        fm = master["configs"]["full_system"]["metrics"]
        for name, outc in outcomes.items():
            if name == "full_system" or master["configs"][name]["metrics"]["status"] != "complete": continue
            m = master["configs"][name]["metrics"]
            master["comparisons_vs_full_system"][name] = {
                "delta_EX_pp": round(m["EX"] - fm["EX"], 2), "mcnemar_EX": mcnemar_exact(full["ex"], outc["ex"]),
                "delta_ESM_pp": round(m["ESM_official"] - fm["ESM_official"], 2), "mcnemar_ESM": mcnemar_exact(full["esm"], outc["esm"]),
            }
        d = {k: v["delta_EX_pp"] for k, v in master["comparisons_vs_full_system"].items()}
        def _sum(keys):
            return round(sum(d[k] for k in keys), 2) if all(k in d for k in keys) else None
        master["redundancy_analysis"] = {
            "definition": "grouped drop vs. sum of the single-component drops of its members (EX pp vs full_system); a grouped drop larger in magnitude than the sum indicates compensation/redundancy among the members",
            "schema_grounding": {"grouped_G1": d.get("grp_no_schema_grounding"), "sum_singles": _sum(["abl_no_column_linking"]), "members": ["abl_no_column_linking"]},
            "value_grounding":  {"grouped_G2": d.get("grp_no_value_grounding"),  "sum_singles": _sum(["abl_no_value_injection", "abl_no_sample_rows"]), "members": ["abl_no_value_injection", "abl_no_sample_rows", "(+ repair-stage value grounding)"]},
            "translation":      {"grouped_G3": d.get("grp_no_translation"),      "sum_singles": None, "members": []},
            "inference_stage":  {"grouped_G4": d.get("greedy_control"),          "sum_singles": _sum(["abl_no_voting", "abl_no_repair"]), "members": ["abl_no_voting", "abl_no_repair"]},
            "all_prompt_and_inference": {"full_minus_baseline_minimal": (-d["baseline_minimal"] if "baseline_minimal" in d else None),
                                         "sum_of_all_single_drops": _sum(["abl_no_column_linking", "abl_no_value_injection", "abl_no_sample_rows", "abl_no_voting", "abl_no_repair"])},
        }
    else:
        master["notes"].append("full_system not complete yet: comparisons and redundancy analysis will be filled in once it is.")
    if write:
        with open(MASTER_JSON_LOCAL, "w", encoding="utf-8") as f:
            json.dump(master, f, ensure_ascii=False, indent=1)
        synced = _push_to_drive(quiet=not verbose)
        if verbose:
            print(f"💾 master JSON → {MASTER_JSON if synced else MASTER_JSON_LOCAL + '  (Drive unavailable — local only)'}"
                  f" ({os.path.getsize(MASTER_JSON_LOCAL)/1e6:.1f} MB)")
    return master

def print_summary(master):
    print(f"\n{LANGUAGE} — {BENCHMARK} — {master['provenance']['suite_samples']} samples")
    print(f"{'config':<26} {'row':<4} {'status':<16} {'EX%':>6} {'EX 95% CI':>16} {'ESM%':>6} {'ESM 95% CI':>16} {'ΔEX':>6} {'p(EX)':>8} {'ΔESM':>6} {'p(ESM)':>8}")
    for name, c in master["configs"].items():
        m = c["metrics"]; cmp = master["comparisons_vs_full_system"].get(name, {})
        ci = lambda v: f"[{v[0]:.1f}, {v[1]:.1f}]" if v else ""
        dx = f"{cmp['delta_EX_pp']:+.1f}" if cmp else ""; px = f"{cmp['mcnemar_EX']['p']:.3g}" if cmp else ""
        de = f"{cmp['delta_ESM_pp']:+.1f}" if cmp else ""; pe = f"{cmp['mcnemar_ESM']['p']:.3g}" if cmp else ""
        print(f"{name:<26} {c['row']:<4} {m['status']:<16} {m['EX'] or 0:6.1f} {ci(m['EX_ci95']):>16} {m['ESM_official'] or 0:6.1f} {ci(m['ESM_ci95']):>16} {dx:>6} {px:>8} {de:>6} {pe:>8}")
    ra = master.get("redundancy_analysis", {})
    if ra:
        print("\nGrouped vs. sum of singles (EX pp vs full_system):")
        for k, v in ra.items():
            if isinstance(v, dict) and "sum_singles" in v:
                g = [x for x in v if x.startswith("grouped")][0]
                print(f"  {k:<26} {g}: {v[g]}   sum of singles: {v['sum_singles']}")
        a = ra.get("all_prompt_and_inference", {})
        print(f"  {'all components':<26} full − baseline: {a.get('full_minus_baseline_minimal')}   sum of all single drops: {a.get('sum_of_all_single_drops')}")

print("✅ Suite helpers ready")


In [ ]:
#@title 🧪 Experiment Suite — Run selected configurations (resumable) { display-mode: "form" }
_suite_t0 = time.time()
_failed = []
for _name in SELECTED:
    try:
        run_config(_name)
    except OSError as _e:
        _failed.append(_name)
        print(f"  ❌ {_name}: local disk error ({_e}) — stopping the suite; restart the runtime and re-run this cell")
        apply_config({})
        break
    except Exception as _e:
        _failed.append(_name)
        print(f"  ❌ {_name} raised {type(_e).__name__}: {_e}")
        traceback.print_exc()
        apply_config({})
    try:
        build_master(write=True, verbose=False)   # keep the master JSON current after every configuration
    except Exception as _e:
        print(f"  (master JSON update skipped: {_e})")
if not _drive_ok():
    print("⚠️ Drive is NOT mounted right now — results are safe in", LOCAL_DIR,
          "but will be lost if the runtime restarts. Remount Drive (Runtime → Restart if needed) and run the Summary cell to push them.")
print("=" * 78)
print(f"Suite pass finished in {(time.time()-_suite_t0)/3600:.2f} h; failed configs: {_failed or 'none'}")
print("Re-run this cell after a disconnect — completed samples are skipped.")


In [ ]:
#@title 🧪 Experiment Suite — Summary & master JSON { display-mode: "form" }
_master = build_master(write=True)
print_summary(_master)
print(f"\nPer-configuration files (local: {LOCAL_DIR}; Drive synced: {_drive_ok()}):")
for _n in CONFIGS:
    _p = os.path.join(LOCAL_DIR, f"{_n}.jsonl")
    if os.path.exists(_p):
        print(f"  {_n:<26} {sum(1 for _ in open(_p, encoding='utf-8'))} samples  {os.path.getsize(_p)/1e6:.1f} MB")
print(f"\nSingle results file for this language: {MASTER_JSON if _drive_ok() else MASTER_JSON_LOCAL}")
try:
    from google.colab import files as _files
    if input("Download the master JSON now? [y/N] ").strip().lower() == "y":
        _files.download(MASTER_JSON_LOCAL)
except Exception:
    pass
